# FC Online 4 데이터 분석 프로젝트

**패키지 출시가 유저 이탈과 매출에 미치는 영향 분석**

이 노트북은 5개의 Python 소스 파일을 하나로 합친 통합 실행 파일입니다.
순서대로 실행하면 데이터 생성 → EDA → ML 모델링 → 시나리오 시뮬레이션까지 전체 파이프라인이 동작합니다.

| 셀 | 내용 | 원본 파일 |
|:---:|:------|:----------|
| 1 | 환경 설정 (한글 폰트 + 스타일) | `font_setup.py` |
| 2 | 데이터 생성 (5개 CSV) | `01_generate_data.py` |
| 3 | EDA 시각화 (18개 차트) | `02_eda_visualization.py` |
| 4 | ML 이탈 예측 모델 | `03_ml_churn_model.py` |
| 5 | 시나리오 시뮬레이션 | `04_scenario_simulation.py` |

In [ ]:
# """
# =============================================================================
# [셀 1] 환경 설정 - 한글 폰트 + 공통 스타일
# =============================================================================
# Colab / Linux / Windows / macOS 자동 감지하여 한글 폰트 설정
# """
# import matplotlib
# import matplotlib.pyplot as plt
# import matplotlib.font_manager as fm
# import platform
# import os
# import subprocess
# import warnings
# import glob

# warnings.filterwarnings('ignore')

# def setup_korean_font():
#     """
#     OS에 맞는 한글 폰트를 자동 감지하여 matplotlib에 적용
#     Colab 환경에서는 폰트 캐시를 삭제하고 재구축
#     Returns: 설정된 폰트 이름 (str)
#     """
#     system = platform.system()
#     font_name = None

#     # ── OS별 기본 한글 폰트 탐색 ──
#     if system == 'Windows':
#         # Windows: 맑은 고딕 → 굴림 → 돋움 순서로 탐색
#         for candidate in ['Malgun Gothic', 'Gulim', 'Dotum', 'Batang']:
#             if any(candidate.lower() in f.name.lower() for f in fm.fontManager.ttflist):
#                 font_name = candidate
#                 break
#         # 시스템 폰트 경로에서 직접 탐색
#         if font_name is None:
#             win_font_dir = os.path.join(os.environ.get('WINDIR', 'C:\\Windows'), 'Fonts')
#             for fname, ffile in [('Malgun Gothic', 'malgun.ttf'), ('Gulim', 'gulim.ttc')]:
#                 fpath = os.path.join(win_font_dir, ffile)
#                 if os.path.exists(fpath):
#                     fm.fontManager.addfont(fpath)
#                     font_name = fname
#                     break

#     elif system == 'Darwin':
#         # macOS: AppleGothic → Apple SD Gothic Neo
#         for candidate in ['AppleGothic', 'Apple SD Gothic Neo']:
#             if any(candidate in f.name for f in fm.fontManager.ttflist):
#                 font_name = candidate
#                 break

#     else:
#         # Linux / Colab: NanumGothic 설치 + 캐시 강제 재구축
#         font_name = 'NanumGothic'
#         if not any(font_name in f.name for f in fm.fontManager.ttflist):
#             print("NanumGothic 폰트 설치 중...")
#             try:
#                 subprocess.run(
#                     ['apt-get', 'install', '-y', 'fonts-nanum'],
#                     capture_output=True, timeout=60
#                 )
#                 print("  → 패키지 설치 완료")
#             except Exception as e:
#                 print(f"  → 패키지 설치 실패: {e}")

#             # ── Colab 핵심: matplotlib 폰트 캐시 강제 삭제 + 재구축 ──
#             cache_dir = matplotlib.get_cachedir()
#             cache_files = glob.glob(os.path.join(cache_dir, 'fontlist-*.json'))
#             for cf in cache_files:
#                 try:
#                     os.remove(cf)
#                     print(f"  → 캐시 삭제: {os.path.basename(cf)}")
#                 except Exception:
#                     pass

#             # 폰트 매니저 재구축 (캐시 무시)
#             fm._load_fontmanager(try_read_cache=False)

#             # .ttf 파일 직접 등록
#             nanum_fonts = glob.glob('/usr/share/fonts/truetype/nanum/*.ttf')
#             for nf in nanum_fonts:
#                 fm.fontManager.addfont(nf)
#                 print(f"  → 폰트 등록: {os.path.basename(nf)}")

#             if nanum_fonts:
#                 print(f"  → NanumGothic 설치 및 등록 완료 ({len(nanum_fonts)}개 폰트)")
#             else:
#                 print("  → NanumGothic ttf 파일을 찾을 수 없습니다")
#                 font_name = 'DejaVu Sans'

#     # 폰트를 찾지 못한 경우 최종 fallback
#     if font_name is None:
#         korean_fonts = [f.name for f in fm.fontManager.ttflist
#                         if any(kw in f.name.lower() for kw in
#                                ['gothic', 'gulim', 'dotum', 'batang', 'nanum', 'malgun', 'apple'])]
#         if korean_fonts:
#             font_name = korean_fonts[0]
#         else:
#             font_name = 'DejaVu Sans'

#     # matplotlib 전역 설정
#     plt.rcParams['font.family'] = font_name
#     plt.rcParams['axes.unicode_minus'] = False
#     plt.rcParams['figure.dpi'] = 150
#     plt.rcParams['savefig.dpi'] = 150
#     plt.rcParams['savefig.bbox'] = 'tight'
#     plt.rcParams['figure.facecolor'] = 'white'
#     plt.rcParams['axes.facecolor'] = 'white'
#     plt.rcParams['axes.grid'] = True
#     plt.rcParams['grid.alpha'] = 0.3

#     return font_name

# # 프로젝트 공통 색상 팔레트
# COLORS = {
#     '0~10조': '#4E79A7',
#     '10~100조': '#F28E2B',
#     '100~1000조': '#59A14F',
#     '1000조이상': '#E15759',
# }

# GROUP_ORDER = ['0~10조', '10~100조', '100~1000조', '1000조이상']

# CHART_STYLE = {
#     'title_fontsize': 14,
#     'label_fontsize': 11,
#     'tick_fontsize': 9,
#     'legend_fontsize': 10,
#     'figsize_single': (10, 6),
#     'figsize_double': (14, 6),
#     'figsize_large': (14, 10),
# }

# # ── 폰트 적용 실행 ──
# font_name = setup_korean_font()
# print(f" 한글 폰트 설정 완료: {font_name}")

# # ── 프로젝트 루트 경로 자동 감지 ──
# # 노트북 위치 기준으로 프로젝트 루트를 찾음
# # 방법: 현재 작업 디렉토리에 'src/' 폴더가 있으면 여기가 프로젝트 루트
# #       없으면 노트북 파일을 기준으로 탐색
# def find_project_root():
#     """프로젝트 루트 디렉토리 자동 탐색"""
#     cwd = os.getcwd()

#     # 1) cwd에 src/ 폴더가 있으면 여기가 프로젝트 루트
#     if os.path.isdir(os.path.join(cwd, 'src')):
#         return cwd

#     # 2) cwd 하위에 fc_online4_analysis/ 가 있는 경우 (Colab /content 등)
#     for sub in ['fc_online4_analysis', 'FC_Online4']:
#         candidate = os.path.join(cwd, sub)
#         if os.path.isdir(os.path.join(candidate, 'src')):
#             return candidate

#     # 3) Colab에서 노트북 업로드 시 /content 하위 탐색
#     if os.path.exists('/content'):
#         for root, dirs, files in os.walk('/content'):
#             if 'src' in dirs and any(f.endswith('.ipynb') for f in files):
#                 return root
#             if root.count(os.sep) - '/content'.count(os.sep) > 2:
#                 break

#     # 4) 최종 fallback: cwd 사용하고 필요한 디렉토리 생성
#     print(f"  프로젝트 루트를 자동 감지하지 못했습니다. cwd 사용: {cwd}")
#     return cwd

# PROJECT_ROOT = find_project_root()
# print(f" 프로젝트 루트: {PROJECT_ROOT}")
# print(f"   데이터 경로: {os.path.join(PROJECT_ROOT, 'data')}")
# print(f"   출력 경로:   {os.path.join(PROJECT_ROOT, 'outputs', 'figures')}")


In [ ]:
"""
=============================================================================
FC Online 4 데이터 분석 프로젝트 - Stage 1: 데이터 생성
=============================================================================
목적: 50,000명의 유저에 대한 5개 테이블(user_profile,login_logs,package_purchase,trade_market,daily_club_value) 생성

=============================================================================
"""

import numpy as np
import pandas as pd
import datetime
import random
import os
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# 1. 전역 설정 및 시드 고정
# ============================================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# 프로젝트 경로 설정
BASE_DIR = PROJECT_ROOT
DATA_DIR = os.path.join(BASE_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

# ============================================================================
# 2. 핵심 파라미터 (설계서 기반)
# ============================================================================
N_USERS = 50000  # 총 유저 수

# 데이터 기간: 2025-01-01 ~ 2025-04-30
DATE_START = datetime.date(2025, 1, 1)
DATE_END = datetime.date(2025, 4, 30)
ALL_DATES = pd.date_range(DATE_START, DATE_END, freq='D')
N_DAYS = len(ALL_DATES)

# 패키지 출시일 (설계서 명시)
PACKAGE_DATES = [
    datetime.date(2025, 1, 18),
    datetime.date(2025, 2, 20),
    datetime.date(2025, 3, 20),
]

# 패키지 종류: Pack A~D, B가 최다 판매 
PACKAGE_IDS = ['Pack_A', 'Pack_B', 'Pack_C', 'Pack_D']
# Pack B가 가장 많이 팔리도록 가중치 설정
PACKAGE_WEIGHTS = [0.20, 0.45, 0.22, 0.13]

# 패키지별 가격 범위 (원 단위)
PACKAGE_PRICES = {
    'Pack_A': (5900, 11900),
    'Pack_B': (11900, 33000),   # 가장 인기 → 중간 가격대
    'Pack_C': (33000, 55000),
    'Pack_D': (55000, 110000),
}

# 그룹 설정 (유저가 수정 요청한 분포)
GROUP_CONFIG = {
    '0~10조': {
        'ratio': 0.07,           # 7%
        'club_value_range': (0.5e12, 10e12),  # 0.5조 ~ 10조
        'ovr_mean': 130.9, 'ovr_std': 2.5,
        'ovr_min': 125, 'ovr_max': 135,
        'daily_login_prob_base': 0.35,   # 기본 접속 확률
        'purchase_prob': 0.15,            # 패키지 구매 확률
        'churn_sensitivity': 0.60,        # 이탈 민감도 (slope)
        'cv_decline_rates': [0.10, 0.17, 0.15],  # 패키지별 구단가치 하락률
        'trade_activity': 0.25,           # 거래 활동 수준
    },
    '10~100조': {
        'ratio': 0.54,           # 54% — 주요 타겟 그룹
        'club_value_range': (10e12, 100e12),
        'ovr_mean': 133.4, 'ovr_std': 1.8,
        'ovr_min': 129, 'ovr_max': 137,
        'daily_login_prob_base': 0.55,
        'purchase_prob': 0.35,
        'churn_sensitivity': 0.80,        # 가장 높은 민감도
        'cv_decline_rates': [0.23, 0.25, 0.26],  # 가장 큰 하락률
        'trade_activity': 0.55,
    },
    '100~1000조': {
        'ratio': 0.35,           # 35%
        'club_value_range': (100e12, 1000e12),
        'ovr_mean': 136.1, 'ovr_std': 1.5,
        'ovr_min': 133, 'ovr_max': 140,
        'daily_login_prob_base': 0.65,
        'purchase_prob': 0.50,
        'churn_sensitivity': 0.25,
        'cv_decline_rates': [0.05, 0.036, 0.03],
        'trade_activity': 0.70,
    },
    '1000조이상': {
        'ratio': 0.04,           # 4%
        'club_value_range': (1000e12, 8000e12),
        'ovr_mean': 137.3, 'ovr_std': 1.2,
        'ovr_min': 135, 'ovr_max': 142,
        'daily_login_prob_base': 0.78,
        'purchase_prob': 0.65,
        'churn_sensitivity': 0.15,
        'cv_decline_rates': [0.01, 0.01, 0.01],
        'trade_activity': 0.85,
    },
}

# 멤버십 등급 및 월 구독료 
# FC Online 실제 월정액 상품 구조를 참고한 설정
MEMBERSHIP_TIERS = ['Bronze', 'Silver', 'Gold', 'Platinum', 'Diamond']
MEMBERSHIP_MONTHLY_FEE = {
    'Bronze':    5900,   # 기본 등급 — 월 5,900원
    'Silver':   11900,   # 실버 등급 — 월 11,900원
    'Gold':     33000,   # 골드 등급 — 월 33,000원
    'Platinum': 55000,   # 플래티넘 등급 — 월 55,000원
    'Diamond': 110000,   # 다이아 등급 — 월 110,000원
}

# 선수 풀 (OVR별 선수 목록 — FC온라인 실제 선수 기반)
PLAYER_POOL = {
    # OVR 125~129: 저OVR 선수
    125: ['김민재_125', '이강인_125', '황의조_125'],
    126: ['손흥민_126', '조규성_126', '정우영_126'],
    127: ['박지성_127', '차범근_127', '이천수_127'],
    128: ['홍명보_128', '안정환_128', '이영표_128'],
    129: ['유상철_129', '기성용_129', '박주영_129'],
    # OVR 130~134: 중OVR 선수 (10~100조 핵심 보유)
    130: ['메시_130', '호날두_130', '네이마르_130', '음바페_130'],
    131: ['벤제마_131', '살라_131', '손흥민_131', '케인_131'],
    132: ['레반도프스키_132', '홀란드_132', '비니시우스_132'],
    133: ['드브라위너_133', '모드리치_133', '크로스_133', '벨링엄_133'],
    134: ['킴미히_134', '카제미루_134', '칸테_134'],
    # OVR 135~137: 고OVR 선수 (100~1000조 핵심 보유)
    135: ['펠레_135', '마라도나_135', '지단_135', '크루이프_135'],
    136: ['베켄바워_136', '호나우두_136', '호나우지뉴_136'],
    137: ['메시_TOTY_137', '음바페_TOTY_137', '홀란드_TOTY_137'],
    # OVR 138~142: 최고OVR (1000조이상 핵심 보유)
    138: ['메시_ICON_138', '펠레_ICON_138'],
    139: ['마라도나_ICON_139', '호나우두_ICON_139'],
    140: ['지단_ICON_140', '베켄바워_ICON_140'],
    141: ['크루이프_ICON_141'],
    142: ['펠레_ULTIMATE_142'],
}

# 패키지에서 높은 확률로 등장하는 OVR ( 확률 높은 OVR → 가격 하락)
# 설계서 버블차트 참조: 40% OVR 133.0, 19% OVR 130.0, 5% OVR 135.0, 1% OVR 136.0
PACKAGE_OVR_PROBS = {
    130: 0.19, 131: 0.12, 132: 0.10, 133: 0.40,
    134: 0.08, 135: 0.05, 136: 0.01,
    # 나머지 OVR은 매우 낮은 확률
    125: 0.01, 126: 0.01, 127: 0.01, 128: 0.01, 129: 0.01,
}

# ============================================================================
# 3. user_profile 생성
# ============================================================================
def generate_user_profile():
    """
    유저 프로필 테이블 생성
    스키마: user_id, nickname, spendig, club_value, membership_tier,
            club_value_group, avg_ovr, register_date, position_player
    """
    print("[1/4] user_profile 생성 중...")

    users = []
    user_id_counter = 100000  # 유저 ID 시작값

    for group_name, config in GROUP_CONFIG.items():
        n_group = int(N_USERS * config['ratio'])

        for i in range(n_group):
            user_id = f"U{user_id_counter}"
            user_id_counter += 1

            # 닉네임 생성 (자연스러운 게임 닉네임)
            nickname = f"Player_{user_id_counter}"

            # 구단가치: 그룹 범위 내에서 로그정규분포로 자연스럽게 생성
            cv_min, cv_max = config['club_value_range']
            log_mean = (np.log(cv_min) + np.log(cv_max)) / 2
            log_std = (np.log(cv_max) - np.log(cv_min)) / 6
            club_value = np.exp(np.random.normal(log_mean, log_std))
            club_value = np.clip(club_value, cv_min * 1.01, cv_max * 0.99)

            # 평균 OVR: 그룹별 평균 ± std, 범위 클리핑
            avg_ovr = np.random.normal(config['ovr_mean'], config['ovr_std'])
            avg_ovr = np.clip(avg_ovr, config['ovr_min'], config['ovr_max'])
            avg_ovr = round(avg_ovr, 1)

            # 누적 과금 상태 (spendig): 구단가치와 상관있게 생성
            # 높은 구단가치 → 높은 과금 경향, but 무과금도 존재
            if group_name == '0~10조':
                # 대부분 소과금 또는 무과금
                if random.random() < 0.3:
                    spendig = 0  # 무과금이지만 데이터에서는 최소값 부여
                else:
                    spendig = int(np.random.lognormal(8, 1.5))
                spendig = max(spendig, int(np.random.uniform(500, 3000)))
            elif group_name == '10~100조':
                # 중과금 핵심 그룹
                spendig = int(np.random.lognormal(10, 1.2))
                spendig = max(spendig, int(np.random.uniform(5000, 20000)))
            elif group_name == '100~1000조':
                spendig = int(np.random.lognormal(11.5, 1.0))
                spendig = max(spendig, int(np.random.uniform(50000, 200000)))
            else:  # 1000조이상
                spendig = int(np.random.lognormal(13, 0.8))
                spendig = max(spendig, int(np.random.uniform(500000, 2000000)))

            # 멤버십 등급: 구단가치 그룹에 따라 현실적 분포 적용
            # 구단가치가 높을수록 고등급 멤버십 비율이 현저히 높음
            if group_name == '0~10조':
                # 거의 무과금: Bronze/Silver 중심
                tier_probs = [0.50, 0.30, 0.12, 0.06, 0.02]
            elif group_name == '10~100조':
                # 중과금: Gold 중심, Silver/Platinum도 분포
                tier_probs = [0.15, 0.28, 0.32, 0.17, 0.08]
            elif group_name == '100~1000조':
                # 고과금: Platinum/Diamond 중심
                tier_probs = [0.03, 0.07, 0.20, 0.35, 0.35]
            else:  # 1000조이상
                # 초고과금 고래: Diamond 압도적
                tier_probs = [0.01, 0.02, 0.05, 0.20, 0.72]
            membership_tier = np.random.choice(MEMBERSHIP_TIERS, p=tier_probs)

            # 가입일: 기간 시작 전 랜덤 (오래된 유저도 있음)
            days_before = random.randint(30, 365)
            register_date = DATE_START - datetime.timedelta(days=days_before)

            # 포지션 선수 (대표 선수 11명 중 랜덤)
            possible_ovrs = [o for o in PLAYER_POOL.keys()
                           if config['ovr_min'] <= o <= config['ovr_max']]
            if possible_ovrs:
                rep_ovr = random.choice(possible_ovrs)
                position_player = random.choice(PLAYER_POOL[rep_ovr])
            else:
                position_player = "Unknown"

            # 월 멤버십 요금: 등급 기반 (데이터에 명시적으로 포함)
            monthly_fee = MEMBERSHIP_MONTHLY_FEE[membership_tier]

            # 월 평균 과금액 (in-game spending): 구단가치 그룹별 현실적 분포
            # 고래 유저일수록 월 과금이 압도적으로 높음 (패키지/선수팩/강화 등)
            if group_name == '0~10조':
                # 거의 무과금: 소액 또는 이벤트성 과금만
                monthly_spending = int(np.clip(
                    np.random.lognormal(8.0, 0.8), 1000, 20000))
            elif group_name == '10~100조':
                # 중과금: 월 1~5만원대 과금
                monthly_spending = int(np.clip(
                    np.random.lognormal(9.1, 0.7), 2000, 80000))
            elif group_name == '100~1000조':
                # 고과금: 월 5~25만원대 과금
                monthly_spending = int(np.clip(
                    np.random.lognormal(10.9, 0.7), 10000, 500000))
            else:  # 1000조이상
                # 초고과금 고래: 월 수백만원 과금 (현질 규모가 다름)
                monthly_spending = int(np.clip(
                    np.random.lognormal(14.3, 0.6), 300000, 10000000))

            users.append({
                'user_id': user_id,
                'nickname': nickname,
                'spendig': spendig,
                'club_value': round(club_value),
                'club_value_group': group_name,
                'avg_ovr': avg_ovr,
                'membership_tier': membership_tier,
                'monthly_membership_fee': monthly_fee,
                'monthly_avg_spending': monthly_spending,
                'register_date': register_date.strftime('%Y-%m-%d'),
                'position_player': position_player,
            })

    # 그룹별 유저 수 합이 50000이 안될 수 있으므로 마지막 그룹에서 보정
    df = pd.DataFrame(users)

    # 부족분을 10~100조 그룹에서 추가 (가장 큰 그룹)
    deficit = N_USERS - len(df)
    if deficit > 0:
        extra_users = []
        config = GROUP_CONFIG['10~100조']
        for i in range(deficit):
            user_id_counter += 1
            cv_min, cv_max = config['club_value_range']
            log_mean = (np.log(cv_min) + np.log(cv_max)) / 2
            log_std = (np.log(cv_max) - np.log(cv_min)) / 6
            club_value = np.exp(np.random.normal(log_mean, log_std))
            club_value = np.clip(club_value, cv_min * 1.01, cv_max * 0.99)
            avg_ovr = round(np.clip(
                np.random.normal(config['ovr_mean'], config['ovr_std']),
                config['ovr_min'], config['ovr_max']
            ), 1)
            spendig = max(int(np.random.lognormal(10, 1.2)),
                         int(np.random.uniform(5000, 20000)))

            tier = np.random.choice(
                MEMBERSHIP_TIERS, p=[0.15, 0.28, 0.32, 0.17, 0.08])
            # 10~100조 그룹 월 평균 과금
            m_spending = int(np.clip(
                np.random.lognormal(9.1, 0.7), 2000, 80000))
            extra_users.append({
                'user_id': f"U{user_id_counter}",
                'nickname': f"Player_{user_id_counter}",
                'spendig': spendig,
                'club_value': round(club_value),
                'club_value_group': '10~100조',
                'avg_ovr': avg_ovr,
                'membership_tier': tier,
                'monthly_membership_fee': MEMBERSHIP_MONTHLY_FEE[tier],
                'monthly_avg_spending': m_spending,
                'register_date': (DATE_START - datetime.timedelta(
                    days=random.randint(30, 365))).strftime('%Y-%m-%d'),
                'position_player': random.choice(
                    PLAYER_POOL.get(133, ['드브라위너_133'])),
            })
        df = pd.concat([df, pd.DataFrame(extra_users)], ignore_index=True)

    # 셔플하여 그룹 순서 섞기
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    print(f"  → user_profile 생성 완료: {len(df)}명")
    print(f"  → 그룹 분포:\n{df['club_value_group'].value_counts().to_string()}")

    return df


# ============================================================================
# 4. login_logs 생성
# ============================================================================
def generate_login_logs(user_profile):
    """
    로그인 로그 테이블 생성
    스키마: user_id, nickname, login_date, session_duration_min

    핵심 로직 (2단계 접근):
    Step A: 각 유저의 이탈 여부 및 이탈 시점을 사전 결정
    Step B: 결정된 이탈 스케줄에 따라 일별 로그인 생성

    이탈률 목표 (설계서 기반, 30일 미접속 기준):
    - 10~100조: 25~35% (가장 높음, sensitivity=0.80)
    - 0~10조: 15~25%
    - 100~1000조: 8~15%
    - 1000조이상: 3~8%
    """
    print("[2/4] login_logs 생성 중...")

    # ── Step A: 이탈 스케줄 사전 결정 ──
    # 그룹별 이탈률 목표 
    CHURN_RATE_TARGET = {
        '0~10조': 0.20,        # 약 20%
        '10~100조': 0.30,      # 약 30% (가장 높음)
        '100~1000조': 0.12,    # 약 12%
        '1000조이상': 0.05,    # 약 5%
    }

    # 이탈 시점 분포: 패키지 출시 후 1~4주 사이에 이탈 집중
    # 각 패키지 출시 후 이탈이 발생하며, 후반 패키지일수록 누적 효과로 이탈 증가
    churn_schedule = {}  # {user_id: churn_date} — 이탈 시작일

    for group_name, config in GROUP_CONFIG.items():
        group_users = user_profile[user_profile['club_value_group'] == group_name]['user_id'].tolist()
        target_rate = CHURN_RATE_TARGET[group_name]
        n_churn = int(len(group_users) * target_rate)

        # 이탈 유저 랜덤 선택
        churn_users = random.sample(group_users, n_churn)

        for uid in churn_users:
            # 이탈 시점: 패키지 출시 후 7~35일 사이 
            # 후반 패키지에서 이탈 확률 더 높음 
            pkg_idx = np.random.choice([0, 1, 2], p=[0.20, 0.35, 0.45])
            pkg_date = PACKAGE_DATES[pkg_idx]
            # 이탈까지의 지연일: 감마 분포로 자연스럽게 (피크 10~20일)
            delay_days = int(np.random.gamma(3, 5)) + 5  # 최소 5일
            delay_days = min(delay_days, 40)  # 최대 40일
            churn_date = pkg_date + datetime.timedelta(days=delay_days)
            # 기간 내로 클리핑
            churn_date = min(churn_date, DATE_END - datetime.timedelta(days=31))
            churn_schedule[uid] = churn_date

    print(f"  → 이탈 예정 유저: {len(churn_schedule):,}명")

    # ── Step B: 일별 로그인 생성 ──
    all_logs = []
    user_nicknames = dict(zip(user_profile['user_id'], user_profile['nickname']))

    for _, user in user_profile.iterrows():
        uid = user['user_id']
        group = user['club_value_group']
        config = GROUP_CONFIG[group]
        base_prob = config['daily_login_prob_base']

        # 유저별 개인 변동성 (±15%)
        personal_factor = np.random.uniform(0.85, 1.15)

        # 이탈 여부 및 시점
        is_churner = uid in churn_schedule
        churn_date = churn_schedule.get(uid, None)

        for date in ALL_DATES:
            current_date = date.date()

            # 기본 접속 확률
            prob_today = base_prob * personal_factor

            # 패키지 출시 이벤트 효과
            for pkg_date in PACKAGE_DATES:
                days_since = (current_date - pkg_date).days
                if days_since == 0:
                    prob_today *= 1.6  # 출시 당일 접속 급증
                elif 1 <= days_since <= 3:
                    prob_today *= 1.3
                elif 4 <= days_since <= 7:
                    prob_today *= 1.1

            # 이탈 유저의 접속 패턴
            if is_churner and current_date >= churn_date:
                days_after_churn = (current_date - churn_date).days
                # 이탈 후 접속 빈도 급감 (지수적 감소)
                # 처음 며칠은 가끔 접속, 이후 거의 접속 안 함
                prob_today = base_prob * 0.15 * np.exp(-days_after_churn / 8)
                prob_today = max(prob_today, 0.003)  # 완전 0은 아님

            # 이탈 직전 접속 빈도 감소 (이탈 예고 패턴)
            if is_churner and churn_date:
                days_to_churn = (churn_date - current_date).days
                if 0 < days_to_churn <= 14:
                    # 이탈 2주 전부터 서서히 접속 감소
                    decay = 0.5 + 0.5 * (days_to_churn / 14)
                    prob_today *= decay

            # 주말 효과
            weekday = current_date.weekday()
            if weekday >= 4:
                prob_today *= np.random.uniform(1.05, 1.18)

            # 확률 클리핑 (극단값 방지)
            prob_today = np.clip(prob_today, 0.002, 0.96)

            # 접속 여부 결정
            if random.random() < prob_today:
                # 세션 시간 (그룹별, 이탈 유저는 짧게)
                if group == '1000조이상':
                    base_session = np.random.lognormal(4.0, 0.7)
                elif group == '100~1000조':
                    base_session = np.random.lognormal(3.8, 0.8)
                elif group == '10~100조':
                    base_session = np.random.lognormal(3.5, 0.9)
                else:
                    base_session = np.random.lognormal(3.0, 1.0)

                session_min = max(2, int(base_session))

                # 이탈 후 접속 시 짧은 세션
                if is_churner and current_date >= churn_date:
                    session_min = max(1, int(session_min * 0.3))

                session_min = min(session_min, 480)

                all_logs.append({
                    'user_id': uid,
                    'nickname': user_nicknames[uid],
                    'login_date': current_date.strftime('%Y-%m-%d'),
                    'session_duration_min': session_min,
                })

    df_logs = pd.DataFrame(all_logs)
    print(f"  → login_logs 생성 완료: {len(df_logs):,}행")

    return df_logs


# ============================================================================
# 5. package_purchase 생성
# ============================================================================
def generate_package_purchase(user_profile):
    """
    패키지 구매 로그 테이블 생성
    스키마: package_id, package_per, user_id, purchase_date, ovr, ovr_qty, player

    핵심 로직:
    - 패키지 출시일 전후 7일에 구매 집중
    - Pack B가 최다 판매 (설계서 Bar Chart)
    - 패키지 내 높은 확률 OVR: 133(40%), 130(19%) 등 (설계서 Bubble Chart)
    - 구매 확률은 그룹별로 차이
    """
    print("[3/4] package_purchase 생성 중...")

    all_purchases = []
    purchase_counter = 0

    for _, user in user_profile.iterrows():
        uid = user['user_id']
        group = user['club_value_group']
        config = GROUP_CONFIG[group]

        base_purchase_prob = config['purchase_prob']

        # 각 패키지 출시 시점에 대해 구매 여부 결정
        for pkg_release_idx, pkg_date in enumerate(PACKAGE_DATES):
            # 출시일 기준 ±7일 구매 윈도우
            for day_offset in range(-2, 8):
                buy_date = pkg_date + datetime.timedelta(days=day_offset)

                # 출시일 당일/다음날에 구매 확률 가장 높음
                if day_offset == 0:
                    day_multiplier = 2.5
                elif day_offset == 1:
                    day_multiplier = 2.0
                elif day_offset == 2:
                    day_multiplier = 1.5
                elif day_offset < 0:
                    day_multiplier = 0.3  # 사전예약 느낌
                else:
                    day_multiplier = max(0.3, 1.0 - (day_offset - 2) * 0.15)

                final_buy_prob = base_purchase_prob * day_multiplier * 0.15
                final_buy_prob = np.clip(final_buy_prob, 0.005, 0.45)

                if random.random() < final_buy_prob:
                    purchase_counter += 1

                    # 패키지 선택 (Pack B 최다)
                    pkg_id = np.random.choice(PACKAGE_IDS, p=PACKAGE_WEIGHTS)

                    # 패키지 가격
                    price_min, price_max = PACKAGE_PRICES[pkg_id]
                    amount = random.randint(price_min, price_max)

                    # 패키지에서 뽑은 OVR (설계서 확률 분포)
                    ovr_options = list(PACKAGE_OVR_PROBS.keys())
                    ovr_probs = list(PACKAGE_OVR_PROBS.values())
                    # 확률 합 정규화
                    total = sum(ovr_probs)
                    ovr_probs = [p / total for p in ovr_probs]

                    obtained_ovr = np.random.choice(ovr_options, p=ovr_probs)

                    # 뽑은 선수 수량 (1~3명, 대부분 1명)
                    ovr_qty = np.random.choice([1, 2, 3], p=[0.65, 0.25, 0.10])

                    # 뽑은 선수 이름
                    if obtained_ovr in PLAYER_POOL:
                        player = random.choice(PLAYER_POOL[obtained_ovr])
                    else:
                        player = f"선수_{obtained_ovr}"

                    # 패키지 확률 (해당 선수의 등장 확률)
                    package_per = round(PACKAGE_OVR_PROBS.get(obtained_ovr, 0.01) * 100, 2)
                    # 자연스러운 변동 추가
                    package_per = round(package_per + np.random.uniform(-2, 2), 2)
                    package_per = max(0.5, min(package_per, 45.0))

                    all_purchases.append({
                        'purchase_id': f"PUR_{purchase_counter:07d}",
                        'package_id': pkg_id,
                        'package_per': package_per,
                        'user_id': uid,
                        'purchase_date': buy_date.strftime('%Y-%m-%d %H:%M:%S'),
                        'amount': amount,
                        'ovr': int(obtained_ovr),
                        'ovr_qty': int(ovr_qty),
                        'player': player,
                    })

    df_purchase = pd.DataFrame(all_purchases)

    # 시간 추가 (자연스러운 구매 시간 분포: 오후~저녁 집중)
    hours = np.random.choice(
        range(24), size=len(df_purchase),
        p=[0.008, 0.005, 0.005, 0.005, 0.005, 0.007,  # 0~5시
           0.01, 0.02, 0.03, 0.04, 0.05, 0.06,        # 6~11시
           0.065, 0.065, 0.07, 0.07, 0.07, 0.065,     # 12~17시
           0.075, 0.075, 0.065, 0.055, 0.04, 0.04]    # 18~23시
    )
    minutes = np.random.randint(0, 60, size=len(df_purchase))
    seconds = np.random.randint(0, 60, size=len(df_purchase))

    df_purchase['purchase_date'] = pd.to_datetime(df_purchase['purchase_date'])
    df_purchase['purchase_date'] = df_purchase['purchase_date'] + \
        pd.to_timedelta(hours, unit='h') + \
        pd.to_timedelta(minutes, unit='m') + \
        pd.to_timedelta(seconds, unit='s')

    df_purchase = df_purchase.sort_values('purchase_date').reset_index(drop=True)

    print(f"  → package_purchase 생성 완료: {len(df_purchase):,}행")
    print(f"  → 패키지별 판매량:\n{df_purchase['package_id'].value_counts().to_string()}")

    return df_purchase


# ============================================================================
# 6. trade_market 생성
# ============================================================================
def generate_trade_market(user_profile):
    """
    이적시장 거래 로그 테이블 생성
    스키마: purchase_id, user_id, trade_date, price_UP, price_LOW,
            price_trade, ovr, player_id, trade_volume

    핵심 로직:
    - 패키지 출시 후 높은확률 OVR 선수의 가격 급락 (공급 증가)
    - 가격 변동이 Z-score 기반 충격 분석 가능하도록 설계
    - OVR별 기본 가격대 설정, 패키지 전/후 가격 변화 반영
    - 설계서 OVR별 라인 그래프 참조: OVR 130 가격 가장 큰 하락
    """
    print("[4/4] trade_market 생성 중...")

    # OVR별 기본 가격 (100억 단위, 설계서 Y축 참조)
    OVR_BASE_PRICES = {
        125: 8,  126: 12,  127: 18,  128: 25,  129: 32,
        130: 42, 131: 55,  132: 65,  133: 75,
        134: 85, 135: 82,  136: 98,
        137: 110, 138: 150, 139: 200, 140: 300, 141: 450, 142: 600,
    }

    # 패키지 출시로 인한 OVR별 가격 하락률 (설계서 라인 그래프)
    # OVR 130: ~30% 하락, OVR 133: ~15% 하락 (확률 높은 순으로 하락 큼)
    PKG_PRICE_DROP = {
        125: 0.03, 126: 0.04, 127: 0.05, 128: 0.08, 129: 0.10,
        130: 0.30, 131: 0.20, 132: 0.15, 133: 0.18,
        134: 0.10, 135: 0.08, 136: 0.05,
        137: 0.03, 138: 0.02, 139: 0.02, 140: 0.01, 141: 0.01, 142: 0.01,
    }

    all_trades = []
    trade_counter = 0

    # 활성 유저 중 거래 참여자 선별
    active_traders = user_profile[
        user_profile['club_value_group'].map(
            lambda g: random.random() < GROUP_CONFIG[g]['trade_activity']
        )
    ]['user_id'].tolist()

    for date in ALL_DATES:
        current_date = date.date()

        # 해당 날짜의 거래 가능한 OVR 목록
        for ovr in range(125, 143):
            if ovr not in OVR_BASE_PRICES:
                continue

            base_price = OVR_BASE_PRICES[ovr]

            # 패키지 출시 효과 계산
            price_modifier = 1.0
            volume_modifier = 1.0

            for pkg_date in PACKAGE_DATES:
                days_since = (current_date - pkg_date).days
                drop_rate = PKG_PRICE_DROP.get(ovr, 0.05)

                if 0 <= days_since <= 2:
                    # 출시 직후: 급격한 가격 하락 + 거래량 급증
                    price_modifier *= (1 - drop_rate * 0.8)
                    volume_modifier *= 3.0
                elif 2 < days_since <= 5:
                    price_modifier *= (1 - drop_rate * 0.6)
                    volume_modifier *= 2.0
                elif 5 < days_since <= 14:
                    # 점진적 회복 but 완전 회복은 안 됨
                    recovery = min(0.5, days_since * 0.03)
                    price_modifier *= (1 - drop_rate * (0.5 - recovery))
                    volume_modifier *= max(1.2, 2.0 - days_since * 0.08)
                elif 14 < days_since <= 30:
                    # 일부 회복
                    price_modifier *= (1 - drop_rate * 0.15)
                    volume_modifier *= 1.1

            # 일별 자연 변동성 추가 (±5%)
            daily_noise = np.random.normal(1.0, 0.03)
            current_price = base_price * price_modifier * daily_noise
            current_price = max(current_price, base_price * 0.3)  # 최저 30%까지만

            # 상한가/하한가 계산
            spread = current_price * np.random.uniform(0.05, 0.15)
            price_up = round(current_price + spread, 2)
            price_low = round(max(current_price - spread, current_price * 0.7), 2)
            price_trade = round(current_price + np.random.uniform(-spread * 0.5, spread * 0.5), 2)

            # 거래가가 상한가/하한가 사이에 있도록 보장
            price_trade = np.clip(price_trade, price_low, price_up)

            # 해당 OVR의 일일 거래량
            base_volume = max(3, int(ovr * 0.5 - 50))  # OVR 높을수록 거래량 적음
            if ovr <= 133:
                base_volume *= 3  # 중간 OVR이 거래량 많음

            daily_volume = max(1, int(base_volume * volume_modifier * np.random.uniform(0.6, 1.4)))

            # 거래 개수: 일일 거래량에 비례 (너무 많으면 샘플링)
            n_trades = min(daily_volume, 15)

            # 선수 선택
            if ovr in PLAYER_POOL:
                available_players = PLAYER_POOL[ovr]
            else:
                available_players = [f"선수_{ovr}"]

            for _ in range(n_trades):
                trade_counter += 1

                # 거래 참여 유저 선택
                trader = random.choice(active_traders) if active_traders else f"U{random.randint(100000, 150000)}"

                # 개별 거래 가격 변동
                individual_noise = np.random.uniform(0.97, 1.03)
                ind_price_trade = round(price_trade * individual_noise, 2)
                ind_price_up = round(price_up * individual_noise, 2)
                ind_price_low = round(price_low * individual_noise, 2)

                # 가격이 0 이하가 되지 않도록 보장
                ind_price_trade = max(ind_price_trade, 0.5)
                ind_price_low = max(ind_price_low, 0.3)
                ind_price_up = max(ind_price_up, ind_price_trade * 1.01)

                player_id = random.choice(available_players)

                all_trades.append({
                    'purchase_id': f"TRD_{trade_counter:08d}",
                    'user_id': trader,
                    'trade_date': current_date.strftime('%Y-%m-%d'),
                    'price_UP': ind_price_up,
                    'price_LOW': ind_price_low,
                    'price_trade': ind_price_trade,
                    'ovr': ovr,
                    'player_id': player_id,
                    'trade_volume': daily_volume,
                })

    df_trade = pd.DataFrame(all_trades)
    print(f"  → trade_market 생성 완료: {len(df_trade):,}행")

    return df_trade


# ============================================================================
# 7. daily_club_value 생성 (유저별 일별 구단가치 변동)
# ============================================================================
def generate_daily_club_value(user_profile, trade_market):
    """
    유저별 일별 구단가치 변동 테이블 생성 (샘플링 + 벡터화 최적화)
    스키마: user_id, date, club_value, club_value_index, club_value_group,
            daily_change_rate, z_score

    핵심 로직:
    - 그룹별 1,000명 샘플링 → 총 4,000명 × 120일 = 48만 행
    - 각 유저의 초기 구단가치에서 시작
    - 패키지 출시 후 그룹별 하락률 적용 (설계서 표 기반)
    - Z-score: 유저별 일별 변동률의 표준화 점수
    - 10~100조 그룹이 가장 큰 하락 경험

    하락률 (설계서 p.10):
      0~10조:    1/18→10%, 2/20→17%, 3/20→15%
      10~100조:  1/18→23%, 2/20→25%, 3/20→26%  (최대)
      100~1000조: 1/18→5%, 2/20→3.6%, 3/20→3%
      1000조이상: 1/18→1%, 2/20→1%, 3/20→1%
    """
    print("[5/5] daily_club_value 생성 중 (샘플링 + 벡터화)...")

    SAMPLE_PER_GROUP = 1000  # 그룹별 샘플 크기

    # 그룹별 패키지 출시 후 하락률 (설계서 기반)
    DECLINE_RATES = {
        '0~10조':      [0.10, 0.17, 0.15],
        '10~100조':    [0.23, 0.25, 0.26],
        '100~1000조':  [0.05, 0.036, 0.03],
        '1000조이상':   [0.01, 0.01, 0.01],
    }

    # 패키지 출시일 → 날짜 인덱스 매핑
    pkg_dates = [
        datetime.date(2025, 1, 18),
        datetime.date(2025, 2, 20),
        datetime.date(2025, 3, 20),
    ]
    date_list = [d.date() for d in ALL_DATES]
    n_days = len(date_list)

    # 패키지 출시 후 일수 매트릭스 (n_days × 3)
    days_since_pkg = np.array([
        [(d - pkg_d).days for pkg_d in pkg_dates]
        for d in date_list
    ])

    all_dfs = []

    for group_name, config in GROUP_CONFIG.items():
        group_users = user_profile[user_profile['club_value_group'] == group_name]

        # 샘플링 (그룹 인원이 SAMPLE_PER_GROUP보다 적으면 전체 사용)
        n_sample = min(SAMPLE_PER_GROUP, len(group_users))
        sampled = group_users.sample(n=n_sample, random_state=SEED)
        user_ids = sampled['user_id'].values
        initial_cvs = sampled['club_value'].values.astype(float)
        declines = DECLINE_RATES[group_name]

        print(f"  → {group_name}: {n_sample}명 샘플링")

        # 벡터화: (n_users, n_days) 매트릭스로 구단가치 변동 계산
        n_users = len(user_ids)

        # 일별 자연 변동 노이즈 (±0.3%)
        daily_noise = np.random.normal(0, 0.003, size=(n_users, n_days))

        # 패키지 충격 효과를 일별 승수(multiplier)로 사전 계산
        pkg_multiplier = np.ones(n_days)
        for pkg_idx in range(3):
            ds = days_since_pkg[:, pkg_idx]  # (n_days,)
            drop = declines[pkg_idx]

            for day_i in range(n_days):
                d = ds[day_i]
                if d == 0:
                    pkg_multiplier[day_i] *= (1 - drop * 0.50)
                elif d == 1:
                    pkg_multiplier[day_i] *= (1 - drop * 0.20)
                elif d == 2:
                    pkg_multiplier[day_i] *= (1 - drop * 0.10)
                elif 3 <= d <= 5:
                    pkg_multiplier[day_i] *= (1 - drop * 0.03)
                elif 6 <= d <= 14:
                    pkg_multiplier[day_i] *= (1 + drop * 0.015)
                elif 15 <= d <= 25:
                    pkg_multiplier[day_i] *= (1 + drop * 0.005)

        # 유저별 개인 변동성 팩터 (±10%)
        personal_factor = np.random.uniform(0.90, 1.10, size=(n_users, 1))

        # 일별 총 승수 = 자연노이즈 + 패키지 충격 (브로드캐스트)
        # noise에 개인 변동성 곱함 → 유저마다 약간 다른 노이즈
        total_daily = (1 + daily_noise * personal_factor) * pkg_multiplier[np.newaxis, :]

        # 누적곱으로 구단가치 시계열 생성
        cv_matrix = initial_cvs[:, np.newaxis] * np.cumprod(total_daily, axis=1)

        # 최소값 보장 (초기값의 30% 이하로 안 떨어지게)
        floor = initial_cvs[:, np.newaxis] * 0.30
        cv_matrix = np.maximum(cv_matrix, floor)

        # DataFrame 구성
        date_strs = [d.strftime('%Y-%m-%d') for d in date_list]
        for u_idx in range(n_users):
            df_user = pd.DataFrame({
                'user_id': user_ids[u_idx],
                'date': date_strs,
                'club_value': np.round(cv_matrix[u_idx]).astype(int),
                'club_value_group': group_name,
            })
            all_dfs.append(df_user)

    df = pd.concat(all_dfs, ignore_index=True)

    # club_value_index 계산 (기준일=100)
    first_day = date_list[0].strftime('%Y-%m-%d')
    baseline = df[df['date'] == first_day].set_index('user_id')['club_value']
    df['baseline_cv'] = df['user_id'].map(baseline)
    df['club_value_index'] = (df['club_value'] / df['baseline_cv']) * 100
    df.drop('baseline_cv', axis=1, inplace=True)

    # daily_change_rate 계산 (전일 대비 변동률 %)
    df = df.sort_values(['user_id', 'date'])
    df['prev_cv'] = df.groupby('user_id')['club_value'].shift(1)
    df['daily_change_rate'] = ((df['club_value'] - df['prev_cv']) / df['prev_cv']) * 100
    # 첫날은 전일 데이터 없으므로 미세 노이즈(-0.05~0.05%)로 채움 (0 방지)
    first_day_mask = df['daily_change_rate'].isna()
    np.random.seed(42)
    df.loc[first_day_mask, 'daily_change_rate'] = np.random.uniform(
        -0.05, 0.05, size=first_day_mask.sum()
    )
    df.drop('prev_cv', axis=1, inplace=True)

    # z_score 계산 (유저별 변동률의 표준화)
    user_stats = df.groupby('user_id')['daily_change_rate'].agg(['mean', 'std']).reset_index()
    user_stats.columns = ['user_id', 'change_mean', 'change_std']
    user_stats['change_std'] = user_stats['change_std'].replace(0, 1)  # 0 방지
    df = df.merge(user_stats, on='user_id', how='left')
    df['z_score'] = (df['daily_change_rate'] - df['change_mean']) / df['change_std']
    df.drop(['change_mean', 'change_std'], axis=1, inplace=True)

    print(f"  → daily_club_value 생성 완료: {len(df):,}행")

    # 그룹별 평균 인덱스 변화 확인
    group_avg = df.groupby(['date', 'club_value_group'])['club_value_index'].mean()
    last_day = date_list[-1].strftime('%Y-%m-%d')
    print(f"  → 최종일 그룹별 평균 인덱스:")
    for group in ['0~10조', '10~100조', '100~1000조', '1000조이상']:
        try:
            val = group_avg.loc[(last_day, group)]
            print(f"    {group}: {val:.1f}")
        except KeyError:
            pass

    return df


# ============================================================================
# 8. 데이터 품질 검증
# ============================================================================
def validate_data(user_profile, login_logs, package_purchase, trade_market):
    """
    생성된 데이터의 품질을 검증
    - 0%, 100% 같은 극단값이 없는지 확인
    - 그룹 분포가 올바른지 확인
    - 결측값 확인
    """
    print("\n" + "="*60)
    print("데이터 품질 검증")
    print("="*60)

    issues = []

    # 1. 그룹 분포 확인
    group_counts = user_profile['club_value_group'].value_counts(normalize=True) * 100
    print(f"\n[검증1] 그룹 분포:")
    for group, pct in group_counts.items():
        print(f"  {group}: {pct:.1f}%")
        if group == '1000조이상' and pct > 5:
            issues.append(f"1000조이상 그룹이 5% 초과: {pct:.1f}%")

    # 2. OVR 범위 확인 (0, 100 같은 극단값 없는지)
    ovr_stats = user_profile.groupby('club_value_group')['avg_ovr'].agg(['mean', 'min', 'max'])
    print(f"\n[검증2] 그룹별 OVR 통계:")
    print(ovr_stats)

    # 3. 이탈률 계산 및 확인
    last_login = login_logs.groupby('user_id')['login_date'].max().reset_index()
    last_login['login_date'] = pd.to_datetime(last_login['login_date'])
    cutoff = pd.Timestamp(DATE_END) - pd.Timedelta(days=30)
    last_login['is_churned'] = last_login['login_date'] < cutoff

    # 그룹별 이탈률
    merged = last_login.merge(user_profile[['user_id', 'club_value_group']], on='user_id')
    churn_by_group = merged.groupby('club_value_group')['is_churned'].mean() * 100
    print(f"\n[검증3] 그룹별 이탈률:")
    for group, rate in churn_by_group.items():
        flag = "⚠️" if rate > 95 or rate < 1 else "✓"
        print(f"  {flag} {group}: {rate:.1f}%")
        if rate > 98 or rate < 0.5:
            issues.append(f"{group} 이탈률 극단값: {rate:.1f}%")

    # 4. 0값 확인
    zero_spending = (user_profile['spendig'] == 0).sum()
    if zero_spending > 0:
        issues.append(f"spendig=0 유저: {zero_spending}명")
        print(f"\n[검증4] ⚠️ spendig=0 유저 {zero_spending}명 발견")
    else:
        print(f"\n[검증4] ✓ spendig=0 유저 없음")

    # 5. 패키지 판매량 검증
    print(f"\n[검증5] 패키지 판매량:")
    print(f"  총 구매 건수: {len(package_purchase):,}")
    print(f"  패키지별:\n{package_purchase['package_id'].value_counts().to_string()}")

    # 6. 거래 가격 0 확인
    zero_price = (trade_market['price_trade'] <= 0).sum()
    if zero_price > 0:
        issues.append(f"거래가 0 이하: {zero_price}건")
    else:
        print(f"\n[검증6] ✓ 거래가 0 이하 없음")

    # 7. 결측값 확인
    for name, df in [('user_profile', user_profile), ('login_logs', login_logs),
                     ('package_purchase', package_purchase), ('trade_market', trade_market)]:
        null_count = df.isnull().sum().sum()
        if null_count > 0:
            issues.append(f"{name}에 결측값 {null_count}개")
            print(f"\n[검증7] ⚠️ {name} 결측값: {null_count}개")

    if issues:
        print(f"\n⚠️ 발견된 이슈 {len(issues)}건:")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print(f"\n✓ 모든 검증 통과!")

    return issues


# ============================================================================
# 8. 메인 실행
# ============================================================================
def main():
    print("="*60)
    print("FC Online 4 데이터 분석 프로젝트 - 데이터 생성")
    print(f"총 유저: {N_USERS:,}명 | 기간: {DATE_START} ~ {DATE_END}")
    print("="*60 + "\n")

    # 1) user_profile 생성
    user_profile = generate_user_profile()

    # 2) login_logs 생성
    login_logs = generate_login_logs(user_profile)

    # 3) package_purchase 생성
    package_purchase = generate_package_purchase(user_profile)

    # 4) trade_market 생성
    trade_market = generate_trade_market(user_profile)

    # 5) daily_club_value 생성 (유저별 일별 구단가치 변동)
    daily_club_value = generate_daily_club_value(user_profile, trade_market)

    # 6) 데이터 품질 검증
    issues = validate_data(user_profile, login_logs, package_purchase, trade_market)

    # 7) CSV 저장
    print("\n" + "="*60)
    print("CSV 파일 저장 중...")

    user_profile.to_csv(os.path.join(DATA_DIR, 'user_profile.csv'), index=False, encoding='utf-8-sig')
    login_logs.to_csv(os.path.join(DATA_DIR, 'login_logs.csv'), index=False, encoding='utf-8-sig')
    package_purchase.to_csv(os.path.join(DATA_DIR, 'package_purchase.csv'), index=False, encoding='utf-8-sig')
    trade_market.to_csv(os.path.join(DATA_DIR, 'trade_market.csv'), index=False, encoding='utf-8-sig')
    daily_club_value.to_csv(os.path.join(DATA_DIR, 'daily_club_value.csv'), index=False, encoding='utf-8-sig')

    print(f"  → {DATA_DIR}/ 에 5개 CSV 저장 완료")

    # 8) 데이터 요약 출력
    print("\n" + "="*60)
    print("데이터 요약")
    print("="*60)
    print(f"  user_profile:      {len(user_profile):>10,}행 × {len(user_profile.columns)}열")
    print(f"  login_logs:        {len(login_logs):>10,}행 × {len(login_logs.columns)}열")
    print(f"  package_purchase:  {len(package_purchase):>10,}행 × {len(package_purchase.columns)}열")
    print(f"  trade_market:      {len(trade_market):>10,}행 × {len(trade_market.columns)}열")
    print(f"  daily_club_value:  {len(daily_club_value):>10,}행 × {len(daily_club_value.columns)}열")

    return user_profile, login_logs, package_purchase, trade_market, daily_club_value


main()

In [ ]:
"""
=============================================================================
FC Online 4 데이터 분석 프로젝트 - EDA 시각화 
=============================================================================
테이블: user_profile, login_logs, package_purchase, trade_market, daily_club_value
=============================================================================
"""

import os
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 폰트 설정 모듈 import
sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings('ignore')
setup_korean_font()

# ============================================================================
# 경로 설정
# ============================================================================
BASE_DIR = PROJECT_ROOT
DATA_PATH = os.path.join(BASE_DIR, 'data')
OUTPUT_PATH = os.path.join(BASE_DIR, 'outputs', 'figures')
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 핵심 파라미터 (설계서 기반)
PACKAGE_DATES = [pd.Timestamp('2025-01-18'),
                 pd.Timestamp('2025-02-20'),
                 pd.Timestamp('2025-03-20')]
PACKAGE_DATE_STRS = ['2025-01-18', '2025-02-20', '2025-03-20']
DATA_END = pd.Timestamp('2025-04-30')
CHURN_DAYS = 30  # 이탈 기준: 30일 미접속

print("=" * 70)
print("FC Online 4 EDA 시각화 — 데이터 기반 전면 재작성")
print("=" * 70)


# ============================================================================
# 데이터 로드
# ============================================================================
def load_all_data():
    """5개 테이블 모두 로드"""
    print("\n[데이터 로드]")
    up = pd.read_csv(os.path.join(DATA_PATH, 'user_profile.csv'))
    ll = pd.read_csv(os.path.join(DATA_PATH, 'login_logs.csv'))
    pp = pd.read_csv(os.path.join(DATA_PATH, 'package_purchase.csv'))
    tm = pd.read_csv(os.path.join(DATA_PATH, 'trade_market.csv'))
    dcv = pd.read_csv(os.path.join(DATA_PATH, 'daily_club_value.csv'))

    # 날짜 변환
    ll['login_date'] = pd.to_datetime(ll['login_date'])
    pp['purchase_date'] = pd.to_datetime(pp['purchase_date'])
    tm['trade_date'] = pd.to_datetime(tm['trade_date'])
    dcv['date'] = pd.to_datetime(dcv['date'])

    for name, df in [('user_profile', up), ('login_logs', ll),
                     ('package_purchase', pp), ('trade_market', tm),
                     ('daily_club_value', dcv)]:
        print(f"  {name}: {len(df):>10,}행")

    return up, ll, pp, tm, dcv


def compute_churn(user_profile, login_logs):
    """이탈 여부 계산 (30일 미접속 기준)"""
    last_login = login_logs.groupby('user_id')['login_date'].max().reset_index()
    last_login.columns = ['user_id', 'last_login']
    last_login['days_inactive'] = (DATA_END - last_login['last_login']).dt.days
    last_login['is_churned'] = last_login['days_inactive'] > CHURN_DAYS

    # monthly_membership_fee, monthly_avg_spending 포함 (하위 호환)
    profile_cols = ['user_id', 'club_value_group', 'spendig',
                    'membership_tier', 'avg_ovr']
    for col in ['monthly_membership_fee', 'monthly_avg_spending']:
        if col in user_profile.columns:
            profile_cols.append(col)
    merged = user_profile[profile_cols].merge(
        last_login, on='user_id', how='left')
    # 로그인 기록 없으면 이탈 처리
    merged['is_churned'] = merged['is_churned'].fillna(True)
    return merged


# ============================================================================
# 차트 공통 유틸리티
# ============================================================================
def add_package_lines(ax, ymin=None, ymax=None, label_y=None):
    """패키지 출시일 수직 점선 추가"""
    pkg_labels = ['Pack 1차\n(1/18)', 'Pack 2차\n(2/20)', 'Pack 3차\n(3/20)']
    for i, (pd_date, label) in enumerate(zip(PACKAGE_DATES, pkg_labels)):
        ax.axvline(pd_date, color='red', linestyle='--', alpha=0.6, linewidth=1)
        if label_y is not None:
            ax.text(pd_date, label_y, label, ha='center', va='bottom',
                    fontsize=7, color='red', alpha=0.8)


def save_fig(fig, filename):
    """차트 저장 — 마운트 경로 길이 제한 우회 (짧은 이름 저장 후 rename)"""
    final_path = os.path.join(OUTPUT_PATH, filename)
    tmp_name = os.path.join(OUTPUT_PATH, '_t.png')
    fig.savefig(tmp_name, bbox_inches='tight', facecolor='white')
    try:
        os.remove(final_path)
    except OSError:
        pass
    os.rename(tmp_name, final_path)
    plt.show()  # 노트북 인라인 출력
    plt.close(fig)
    print(f"  → 저장: {filename}")


# ============================================================================
# Fig 01: K-Means 엘보우 메서드 (k=4 최적)
# ============================================================================
def fig_01_kmeans_elbow(user_profile):
    """실제 user_profile의 club_value + avg_ovr + spendig로 K-means 엘보우"""
    print("\n[Fig 01] K-Means 엘보우")

    # 클러스터링 피처 준비
    features = user_profile[['club_value', 'avg_ovr', 'spendig']].copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(features)

    # k=2~8에 대해 inertia 계산
    K_range = range(2, 9)
    inertias = []
    for k in K_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(X)
        inertias.append(km.inertia_)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)

    # k=4에서 강조 표시
    k4_idx = 2  # k=4는 인덱스 2
    ax.plot(4, inertias[k4_idx], 'r*', markersize=20, zorder=5)
    ax.annotate(f'최적 k=4\n(inertia={inertias[k4_idx]:,.0f})',
                xy=(4, inertias[k4_idx]),
                xytext=(5.5, inertias[k4_idx] * 1.1),
                fontsize=11, color='red', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

    ax.set_xlabel('클러스터 수 (k)', fontsize=12)
    ax.set_ylabel('Inertia (군집 내 거리 합)', fontsize=12)
    ax.set_title('K-Means 엘보우 메서드 — 최적 클러스터 수 결정', fontsize=14)
    ax.set_xticks(list(K_range))
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'fig_01_kmeans_elbow.png')


# ============================================================================
# Fig 02: 그룹별 유저 분포 (파이차트 + 바차트)
# ============================================================================
def fig_02_group_distribution(user_profile):
    """그룹별 유저 수 분포"""
    print("\n[Fig 02] 그룹별 유저 분포")

    counts = user_profile['club_value_group'].value_counts()
    counts = counts.reindex(GROUP_ORDER)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # 파이차트
    colors = [COLORS[g] for g in GROUP_ORDER]
    wedges, texts, autotexts = ax1.pie(
        counts.values, labels=GROUP_ORDER, colors=colors,
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
    for at in autotexts:
        at.set_fontweight('bold')
    ax1.set_title('구단가치 그룹별 유저 비율', fontsize=13)

    # 바차트
    bars = ax2.bar(GROUP_ORDER, counts.values, color=colors, edgecolor='white')
    for bar, val in zip(bars, counts.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}명', ha='center', fontsize=10, fontweight='bold')
    ax2.set_ylabel('유저 수', fontsize=12)
    ax2.set_title('구단가치 그룹별 유저 수', fontsize=13)
    ax2.grid(axis='y', alpha=0.3)

    fig.suptitle('FC Online 4 유저 분포 분석', fontsize=15, fontweight='bold', y=1.02)
    fig.tight_layout()
    save_fig(fig, 'fig_02_group_distribution.png')


# ============================================================================
# Fig 02-b: 그룹별 보유 OVR 바이올린 플롯
# ============================================================================
def fig_02_ovr_violin(user_profile, daily_club_value):
    """그룹별 보유 선수 OVR 분포를 바이올린 플롯으로 시각화

    - 각 그룹의 OVR 분포 형태(중앙값, 분산)를 비교
    - 패키지 높은확률 OVR 구간(130~134) 하이라이트
    - 중위값(밀집 구간) OVR 수치를 명시적으로 표시
    """
    print("\n[Fig 02-b] 그룹별 OVR 바이올린 플롯")

    # user_profile에서 그룹별 대표 OVR 추출 (avg_ovr 컬럼 사용)
    df = user_profile[['user_id', 'club_value_group', 'avg_ovr']].copy()
    df['club_value_group'] = pd.Categorical(
        df['club_value_group'], categories=GROUP_ORDER, ordered=True
    )

    fig, ax = plt.subplots(figsize=(14, 7))

    # 그룹별 데이터 분리
    group_data = [df[df['club_value_group'] == g]['avg_ovr'].dropna().values
                  for g in GROUP_ORDER]

    # 바이올린 플롯 (박스플롯 내장으로 더 풍성하게)
    parts = ax.violinplot(group_data, positions=range(len(GROUP_ORDER)),
                          showmeans=False, showmedians=False, showextrema=False,
                          widths=0.75)

    # 색상 적용 — 그룹별 고유색상
    colors_list = [COLORS[g] for g in GROUP_ORDER]
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors_list[i])
        pc.set_alpha(0.55)
        pc.set_edgecolor(colors_list[i])
        pc.set_linewidth(1.5)

    # 바이올린 위에 박스플롯을 겹쳐서 사분위수 시각화
    bp = ax.boxplot(group_data, positions=range(len(GROUP_ORDER)),
                    widths=0.12, patch_artist=True,
                    showfliers=False, zorder=3,
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(color='gray', linewidth=1),
                    capprops=dict(color='gray', linewidth=1))
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(colors_list[i])
        patch.set_alpha(0.9)
        patch.set_edgecolor('white')
        patch.set_linewidth(1.5)

    # ── 핵심: 중위값(밀집 구간) OVR 수치 표시 ──
    for i, (g, data) in enumerate(zip(GROUP_ORDER, group_data)):
        median_val = np.median(data)
        mean_val = np.mean(data)
        q25 = np.percentile(data, 25)
        q75 = np.percentile(data, 75)

        # 중위값 수치 표시 (큰 흰색 텍스트 + 배경)
        ax.annotate(f'{median_val:.1f}',
                    xy=(i, median_val), xytext=(i + 0.38, median_val),
                    fontsize=12, fontweight='bold', color=colors_list[i],
                    va='center', ha='left',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                              edgecolor=colors_list[i], alpha=0.9, linewidth=1.5),
                    arrowprops=dict(arrowstyle='->', color=colors_list[i],
                                    lw=1.2))

        # 평균값 다이아몬드 마커
        ax.scatter(i, mean_val, marker='D', color='black', s=30, zorder=5,
                   label='평균' if i == 0 else None)

        # IQR 범위 텍스트 (25%~75% 구간)
        ax.text(i, ax.get_ylim()[0] + 0.5, f'IQR: {q25:.1f}~{q75:.1f}',
                ha='center', fontsize=8, color='gray', style='italic')

    # 패키지 높은확률 OVR 구간 하이라이트 (130~134)
    ax.axhspan(130, 134, alpha=0.08, color='red', zorder=0)
    ax.axhline(133, color='red', linestyle='--', alpha=0.5, linewidth=1.2,
               label='OVR 133 (패키지 최고확률 40%)')
    ax.axhline(130, color='red', linestyle=':', alpha=0.3, linewidth=1,
               label='OVR 130 (패키지 확률 19%)')

    # 우측에 패키지 OVR 구간 텍스트 레이블
    ax.text(len(GROUP_ORDER) - 0.55, 133.3, '← 패키지 OVR 133 (40%)',
            fontsize=9, color='red', alpha=0.7, va='bottom')
    ax.text(len(GROUP_ORDER) - 0.55, 130.3, '← 패키지 OVR 130 (19%)',
            fontsize=9, color='red', alpha=0.7, va='bottom')

    ax.set_xticks(range(len(GROUP_ORDER)))
    ax.set_xticklabels(GROUP_ORDER, fontsize=12, fontweight='bold')
    ax.set_ylabel('보유 선수 평균 OVR', fontsize=13)
    ax.set_title('그룹별 보유 선수 OVR 분포 — 중위값(밀집 구간) 및 패키지 OVR 비교',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
    ax.grid(axis='y', alpha=0.2, linestyle='-')

    # Y축 범위 여유
    y_min = min(np.min(d) for d in group_data) - 2
    y_max = max(np.max(d) for d in group_data) + 2
    ax.set_ylim(y_min, y_max)

    fig.tight_layout()
    save_fig(fig, 'fig_02_ovr_violin.png')


# ============================================================================
# Fig 03: 패키지 OVR 확률 버블차트
# ============================================================================
def fig_03_package_ovr_bubble(package_purchase):
    """실제 패키지 구매 데이터에서 OVR별 등장 빈도를 버블로 표시"""
    print("\n[Fig 03] 패키지 OVR 버블차트")

    # OVR별 등장 횟수 및 평균 가격 계산
    ovr_stats = package_purchase.groupby('ovr').agg(
        count=('ovr', 'size'),
        avg_price=('amount', 'mean')
    ).reset_index()

    # 등장 비율 계산
    ovr_stats['pct'] = ovr_stats['count'] / ovr_stats['count'].sum() * 100

    fig, ax = plt.subplots(figsize=(12, 7))

    # 버블 크기: 비율에 비례 (최소 크기 보장)
    bubble_sizes = ovr_stats['pct'] * 120  # 스케일 조정
    bubble_sizes = bubble_sizes.clip(lower=50)

    scatter = ax.scatter(
        ovr_stats['ovr'], ovr_stats['avg_price'],
        s=bubble_sizes, alpha=0.7,
        c=ovr_stats['pct'], cmap='YlOrRd',
        edgecolors='black', linewidths=0.5)

    # 주요 OVR 레이블 추가
    for _, row in ovr_stats.iterrows():
        if row['pct'] >= 3:  # 3% 이상만 레이블
            ax.annotate(f"OVR {int(row['ovr'])}\n({row['pct']:.1f}%)",
                        xy=(row['ovr'], row['avg_price']),
                        fontsize=9, ha='center', va='bottom',
                        fontweight='bold')

    cbar = fig.colorbar(scatter, ax=ax, label='등장 비율 (%)')
    ax.set_xlabel('OVR (선수 능력치)', fontsize=12)
    ax.set_ylabel('평균 패키지 가격 (원)', fontsize=12)
    ax.set_title('패키지에서 OVR별 등장 확률 및 평균 가격', fontsize=14)
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'fig_03_ovr_bubble.png')


# ============================================================================
# Fig 04: 그룹별 OVR 분포 (바이올린)
# ============================================================================
# ============================================================================
# Fig 05: 패키지별 판매량 비교
# ============================================================================
def fig_05_package_sales(package_purchase):
    """패키지 종류별 판매량 + 매출"""
    print("\n[Fig 05] 패키지별 판매량")

    pkg_stats = package_purchase.groupby('package_id').agg(
        count=('package_id', 'size'),
        revenue=('amount', 'sum')
    ).reindex(['Pack_A', 'Pack_B', 'Pack_C', 'Pack_D'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    colors_pkg = ['#6baed6', '#fd8d3c', '#74c476', '#9e9ac8']

    # 판매량
    bars1 = ax1.bar(pkg_stats.index, pkg_stats['count'], color=colors_pkg, edgecolor='white')
    for bar, val in zip(bars1, pkg_stats['count']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', fontsize=10, fontweight='bold')
    ax1.set_ylabel('판매 건수', fontsize=12)
    ax1.set_title('패키지 유형별 판매량', fontsize=13)
    ax1.grid(axis='y', alpha=0.3)

    # 매출
    bars2 = ax2.bar(pkg_stats.index, pkg_stats['revenue'] / 1e6,
                    color=colors_pkg, edgecolor='white')
    for bar, val in zip(bars2, pkg_stats['revenue'] / 1e6):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val:,.0f}백만', ha='center', fontsize=9, fontweight='bold')
    ax2.set_ylabel('매출 (백만 원)', fontsize=12)
    ax2.set_title('패키지 유형별 매출', fontsize=13)
    ax2.grid(axis='y', alpha=0.3)

    fig.suptitle('패키지 판매 분석 — Pack B 최다 판매', fontsize=15, fontweight='bold', y=1.02)
    fig.tight_layout()
    save_fig(fig, 'fig_05_package_sales.png')


# ============================================================================
# Fig 07: 구단가치 인덱스 시계열 (daily_club_value 기반)
# ============================================================================
def fig_07_club_value_index(dcv):
    """daily_club_value에서 그룹별 평균 인덱스 시계열"""
    print("\n[Fig 07] 구단가치 인덱스 시계열")

    # 그룹×날짜별 평균 인덱스
    group_daily = dcv.groupby(['date', 'club_value_group'])['club_value_index'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(14, 7))

    for group in GROUP_ORDER:
        gdata = group_daily[group_daily['club_value_group'] == group]
        ax.plot(gdata['date'], gdata['club_value_index'],
                color=COLORS[group], linewidth=2, label=group)

    # 패키지 출시일 표시
    add_package_lines(ax, label_y=105)

    ax.axhline(100, color='gray', linestyle=':', alpha=0.5, linewidth=1)
    ax.set_xlabel('날짜', fontsize=12)
    ax.set_ylabel('구단가치 인덱스 (기준일=100)', fontsize=12)
    ax.set_title('그룹별 구단가치 인덱스 변화 — 10~100조 그룹 최대 하락', fontsize=14)
    ax.legend(fontsize=11, loc='lower left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.xticks(rotation=45)
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'fig_07_club_value_index.png')


# ============================================================================
# Fig 08: OVR별 이적시장 가격 변화 (trade_market 기반)
# ============================================================================
def fig_08_ovr_price_trend(trade_market):
    """trade_market에서 OVR별 일별 평균 거래 가격 시계열"""
    print("\n[Fig 08] OVR별 가격 변화")

    # 주요 OVR만 표시 (설계서 기준: 130, 133, 135, 136)
    target_ovrs = [130, 133, 135, 136]
    ovr_colors = {130: '#e41a1c', 133: '#377eb8', 135: '#4daf4a', 136: '#984ea3'}

    fig, ax = plt.subplots(figsize=(14, 7))

    for ovr_val in target_ovrs:
        ovr_data = trade_market[trade_market['ovr'] == ovr_val]
        daily_avg = ovr_data.groupby('trade_date')['price_trade'].mean().reset_index()
        # 7일 이동평균으로 스무딩
        daily_avg['price_ma7'] = daily_avg['price_trade'].rolling(7, min_periods=1).mean()
        ax.plot(daily_avg['trade_date'], daily_avg['price_ma7'],
                color=ovr_colors[ovr_val], linewidth=2,
                label=f'OVR {ovr_val}')

    add_package_lines(ax, label_y=ax.get_ylim()[1] * 0.95)

    ax.set_xlabel('날짜', fontsize=12)
    ax.set_ylabel('평균 거래가 (100억 단위)', fontsize=12)
    ax.set_title('주요 OVR 선수 이적시장 가격 변동 (7일 이동평균)', fontsize=14)
    ax.legend(fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.xticks(rotation=45)
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'fig_08_ovr_price_trend.png')


# ============================================================================
# Fig 09: DID (이중차분법) 분석 — daily_club_value + login_logs
# ============================================================================
def fig_09_did_analysis(dcv, churn_data):
    """
    DID (이중차분법): Treatment=10~100조(가장 민감) vs Control=1000조이상(가장 안정)
    기간: 1차 패키지 출시(1/18) 전/후로 구단가치 인덱스 변화 비교
    → Treatment 그룹이 패키지 출시 후 훨씬 큰 하락을 경험
    """
    print("\n[Fig 09] DID 분석")

    treatment_group = '10~100조'
    control_group = '1000조이상'
    cutoff = pd.Timestamp('2025-01-18')  # 1차 패키지 출시일

    # 그룹별 일별 평균 인덱스
    group_daily = dcv.groupby(['date', 'club_value_group'])['club_value_index'].mean().reset_index()

    # Pre / Post 분리
    treat_data = group_daily[group_daily['club_value_group'] == treatment_group]
    ctrl_data = group_daily[group_daily['club_value_group'] == control_group]

    fig, ax = plt.subplots(figsize=(14, 7))

    # Treatment (10~100조)
    ax.plot(treat_data['date'], treat_data['club_value_index'],
            color=COLORS[treatment_group], linewidth=2.5,
            label=f'Treatment ({treatment_group})')

    # Control (1000조이상)
    ax.plot(ctrl_data['date'], ctrl_data['club_value_index'],
            color=COLORS[control_group], linewidth=2.5,
            label=f'Control ({control_group})')

    # 패키지 출시일 전/후 영역 표시
    ax.axvline(cutoff, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvspan(dcv['date'].min(), cutoff, alpha=0.05, color='green', label='Pre (출시 전)')
    ax.axvspan(cutoff, dcv['date'].max(), alpha=0.05, color='red', label='Post (출시 후)')

    # DID 수치 계산 (1차 패키지 기준)
    pre_treat = treat_data[treat_data['date'] < cutoff]['club_value_index'].mean()
    post_treat = treat_data[treat_data['date'] >= cutoff]['club_value_index'].mean()
    pre_ctrl = ctrl_data[ctrl_data['date'] < cutoff]['club_value_index'].mean()
    post_ctrl = ctrl_data[ctrl_data['date'] >= cutoff]['club_value_index'].mean()

    did_effect = (post_treat - pre_treat) - (post_ctrl - pre_ctrl)

    # DID 효과 텍스트
    ax.text(0.02, 0.15, f'DID 효과 = {did_effect:.2f}\n'
            f'Treatment(출시후-전): {post_treat - pre_treat:.2f}\n'
            f'Control(출시후-전): {post_ctrl - pre_ctrl:.2f}',
            transform=ax.transAxes, fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            verticalalignment='bottom')

    add_package_lines(ax, label_y=103)
    ax.axhline(100, color='gray', linestyle=':', alpha=0.4)

    ax.set_xlabel('날짜', fontsize=12)
    ax.set_ylabel('구단가치 인덱스 (기준일=100)', fontsize=12)
    ax.set_title('DID 분석: Treatment(10~100조) vs Control(1000조이상) 구단가치 변화', fontsize=14)
    ax.legend(fontsize=10, loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.xticks(rotation=45)
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'fig_09_did_analysis.png')


# ============================================================================
# Fig 10: 그룹별 이탈률 바차트
# ============================================================================
def fig_10_churn_by_group(churn_data):
    """그룹별 이탈률 (실제 login_logs 기반)"""
    print("\n[Fig 10] 그룹별 이탈률")

    churn_rate = churn_data.groupby('club_value_group')['is_churned'].mean() * 100
    churn_rate = churn_rate.reindex(GROUP_ORDER)

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = [COLORS[g] for g in GROUP_ORDER]
    bars = ax.bar(GROUP_ORDER, churn_rate.values, color=colors, edgecolor='white')

    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

    ax.set_ylabel('이탈률 (%)', fontsize=12)
    ax.set_title('구단가치 그룹별 이탈률 (30일 미접속 기준)', fontsize=14)
    ax.grid(axis='y', alpha=0.3)

    # 10~100조 강조
    ax.annotate('가장 높은 이탈률', xy=(1, churn_rate['10~100조']),
                xytext=(2.2, churn_rate['10~100조'] + 3),
                fontsize=11, color='red', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red'))

    save_fig(fig, 'fig_10_churn_by_group.png')


# ============================================================================
# Fig 11: 구단가치 하락률 vs 이탈률 민감도 (Scatter)
# ============================================================================
def fig_11_sensitivity_analysis(dcv, churn_data):
    """
    X축: 그룹별 구단가치 평균 하락률 (daily_club_value에서 계산)
    Y축: 그룹별 이탈률 (login_logs에서 계산)
    → 민감도 = 기울기
    """
    print("\n[Fig 11] 민감도 분석")

    # 그룹별 평균 구단가치 하락률 계산
    first_date = dcv['date'].min()
    last_date = dcv['date'].max()

    first_val = dcv[dcv['date'] == first_date].groupby('club_value_group')['club_value'].mean()
    last_val = dcv[dcv['date'] == last_date].groupby('club_value_group')['club_value'].mean()
    decline_pct = ((first_val - last_val) / first_val * 100).reindex(GROUP_ORDER)

    # 그룹별 이탈률
    churn_rate = churn_data.groupby('club_value_group')['is_churned'].mean() * 100
    churn_rate = churn_rate.reindex(GROUP_ORDER)

    fig, ax = plt.subplots(figsize=(10, 7))

    for g in GROUP_ORDER:
        ax.scatter(decline_pct[g], churn_rate[g],
                   s=200, color=COLORS[g], edgecolors='black',
                   linewidths=1, zorder=5)
        ax.annotate(f'{g}\n(하락 {decline_pct[g]:.1f}%, 이탈 {churn_rate[g]:.1f}%)',
                    xy=(decline_pct[g], churn_rate[g]),
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=10, fontweight='bold', color=COLORS[g])

    # 추세선
    x_vals = decline_pct.values
    y_vals = churn_rate.values
    z = np.polyfit(x_vals, y_vals, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x_vals.min() - 2, x_vals.max() + 2, 100)
    ax.plot(x_line, p(x_line), '--', color='gray', alpha=0.7, linewidth=1.5,
            label=f'추세선 (기울기={z[0]:.2f})')

    ax.set_xlabel('구단가치 평균 하락률 (%)', fontsize=12)
    ax.set_ylabel('이탈률 (%)', fontsize=12)
    ax.set_title('구단가치 하락 vs 이탈률 민감도 분석\n10~100조 그룹이 가장 민감', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'fig_11_sensitivity.png')


# ============================================================================
# Fig 12: 월별 매출 vs 이탈 손실 (monthly_membership 방식)
# ============================================================================
def fig_12_revenue_vs_loss(user_profile, churn_data, package_purchase):
    """
    매출: 패키지 판매 합계
    손실: 이탈유저의 실제 월멤버십 요금 × 8개월 (데이터 기반)
    → 손실이 매출보다 약간 더 크게
    """
    print("\n[Fig 12] 매출 vs 이탈 손실")

    # 매출: 그룹별 패키지 매출
    pkg_with_group = package_purchase.merge(
        user_profile[['user_id', 'club_value_group']], on='user_id', how='left')
    group_revenue = pkg_with_group.groupby('club_value_group')['amount'].sum()
    group_revenue = group_revenue.reindex(GROUP_ORDER).fillna(0)

    # 손실: 이탈 유저의 (멤버십 + 월 과금) 합산 × 8개월
    # 유저 이탈 시 멤버십뿐 아니라 월 과금(패키지/강화/선수팩 등)도 전부 손실
    LOSS_MONTHS = 8
    churned_users = churn_data[churn_data['is_churned']]

    group_loss = pd.Series({
        g: (churned_users[churned_users['club_value_group'] == g][
            'monthly_membership_fee'].sum()
            + churned_users[churned_users['club_value_group'] == g][
            'monthly_avg_spending'].sum()) * LOSS_MONTHS
        for g in GROUP_ORDER
    })

    fig, ax = plt.subplots(figsize=(12, 7))

    x = np.arange(len(GROUP_ORDER))
    width = 0.35

    # 억 단위 변환
    rev_ok = group_revenue.values / 1e8
    loss_ok = group_loss.values / 1e8

    bars1 = ax.bar(x - width/2, rev_ok, width, label='패키지 매출',
                   color='#2ecc71', edgecolor='white')
    bars2 = ax.bar(x + width/2, loss_ok, width, label='이탈 손실',
                   color='#e74c3c', edgecolor='white')

    for bar, val in zip(bars1, rev_ok):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}억', ha='center', fontsize=9, fontweight='bold')
    for bar, val in zip(bars2, loss_ok):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}억', ha='center', fontsize=9, fontweight='bold', color='red')

    ax.set_xlabel('구단가치 그룹', fontsize=12)
    ax.set_ylabel('금액 (억 원)', fontsize=12)
    ax.set_title('그룹별 패키지 매출 vs 이탈로 인한 손실', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(GROUP_ORDER, fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)

    # 각주: 손실 산정 기준 명시
    fig.text(0.5, -0.02,
             f'※ 이탈 손실 = 이탈 유저 × (월 멤버십 + 월 과금) × {LOSS_MONTHS}개월 (향후 기대수익 손실)',
             ha='center', fontsize=9, style='italic', color='gray')

    save_fig(fig, 'fig_12_revenue_vs_loss.png')


# ============================================================================
# Fig 12b: 10~100조 그룹 이탈이 왜 치명적인가 — 매출 기여도 + 손실 집중도 분석
# ============================================================================
def fig_12b_why_mid_group_matters(user_profile, churn_data, package_purchase):
    """
    패키지 출시로 인한 이탈이 게임사 영업이익에 치명적인 이유를 시각화:
      핵심 논리:
        - 1000조이상: 1인당 매출은 압도적 (고과금 유저)
        - 그러나 10~100조(54%) + 100~1000조(35%) = 전체 유저의 89%
        - 패키지 출시 → 이 89%에서 대규모 이탈 발생
        - 소수 고과금 유저의 매출로는 다수 중간층 이탈 손실을 메울 수 없음
        - 결과: 게임사 전체 영업이익 감소

      (좌상) 1인당 매출(ARPU) vs 유저 비중 — 1000조이상은 ARPU 최고지만 유저 4%
      (우상) 그룹별 이탈 유저 수 + 유저 비중 — 89%의 핵심 유저층이 이탈 타격
      (좌하) 그룹별 이탈 손실 금액 — 이탈자 수 × 실제 월 요금이 합쳐지면 중간층 손실 최대
      (우하) 영업이익 관점 종합 — 패키지 매출 vs 핵심 유저층(89%) 이탈 손실
    """
    print("\n[Fig 12b] 패키지 출시가 게임사 영업이익에 미치는 영향")

    # ── 공통 데이터 준비 ──
    LOSS_MONTHS = 8

    # 그룹별 유저 수 및 비율
    group_counts = user_profile.groupby('club_value_group').size().reindex(GROUP_ORDER)
    total_users = group_counts.sum()
    group_pct = group_counts / total_users * 100
    core_pct = group_pct['10~100조'] + group_pct['100~1000조']  # 핵심 유저층 비중

    # 그룹별 월 기대수익 = 멤버십 + 월 과금 (실제 데이터 기반)
    group_avg_fee = user_profile.groupby('club_value_group')['monthly_membership_fee'].mean()
    group_avg_spending = user_profile.groupby('club_value_group')['monthly_avg_spending'].mean()
    group_avg_rev = (group_avg_fee + group_avg_spending).reindex(GROUP_ORDER)
    group_avg_fee = group_avg_fee.reindex(GROUP_ORDER)
    group_avg_spending = group_avg_spending.reindex(GROUP_ORDER)
    print(f"  그룹별 월 기대수익 (멤버십 + 과금, 데이터 기반):")
    for g in GROUP_ORDER:
        print(f"    {g}: 멤버십 {group_avg_fee[g]:,.0f} + 과금 {group_avg_spending[g]:,.0f} = {group_avg_rev[g]:,.0f}원/월 (유저 {group_counts[g]:,}명, {group_pct[g]:.0f}%)")
    print(f"  → 10~100조 + 100~1000조 = 전체 유저의 {core_pct:.0f}%")

    # 패키지 매출 (그룹별)
    pkg_with_group = package_purchase.merge(
        user_profile[['user_id', 'club_value_group']], on='user_id', how='left')
    pkg_revenue = pkg_with_group.groupby('club_value_group')['amount'].sum()
    pkg_revenue = pkg_revenue.reindex(GROUP_ORDER).fillna(0)

    # ARPU (만원 단위) — 월 기대수익(멤버십+과금) 기준
    arpu = group_avg_rev / 1e4

    # 이탈 데이터
    churned_users = churn_data[churn_data['is_churned']]
    churned_count = churned_users.groupby('club_value_group').size().reindex(GROUP_ORDER).fillna(0)
    total_count = churn_data.groupby('club_value_group').size().reindex(GROUP_ORDER)
    churn_rate = (churned_count / total_count * 100)

    # 이탈 손실 (억 단위) — (멤버십 + 월 과금) × 8개월
    group_loss = pd.Series({
        g: (churned_users[churned_users['club_value_group'] == g][
            'monthly_membership_fee'].sum()
            + churned_users[churned_users['club_value_group'] == g][
            'monthly_avg_spending'].sum()) * LOSS_MONTHS / 1e8
        for g in GROUP_ORDER
    })

    # 총 매출 (패키지 + 멤버십 + 과금)
    group_total_rev = user_profile.groupby('club_value_group').apply(
        lambda x: (x['monthly_membership_fee'].sum() + x['monthly_avg_spending'].sum())
    ).reindex(GROUP_ORDER)
    revenue_4m = group_total_rev * 4 / 1e8  # 4개월
    pkg_rev_ok = pkg_revenue / 1e8
    total_rev = pkg_rev_ok + revenue_4m

    # ── 시각화 (2×2 레이아웃) ──
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    group_colors = [COLORS[g] for g in GROUP_ORDER]
    core_color = '#e74c3c'   # 핵심 유저층(10~100조+100~1000조) 강조색
    whale_color = '#3498db'  # 1000조이상 강조색

    # ═══════════════════════════════════════════════════
    # (좌상) ① 1인당 매출(ARPU) vs 유저 비중 — 역전 현상
    # ═══════════════════════════════════════════════════
    ax1 = axes[0, 0]

    # 이중 Y축: 막대=ARPU, 선=유저비중
    bars = ax1.bar(GROUP_ORDER, arpu.values, color=group_colors,
                   edgecolor='white', linewidth=1.5, alpha=0.85)
    for bar, val in zip(bars, arpu.values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}만', ha='center', fontsize=10, fontweight='bold')

    ax1.set_ylabel('ARPU — 인당 월 기대수익 (만원)', fontsize=11)
    ax1.set_title('① 1인당 기대수익은 1000조이상이 압도적이지만...', fontsize=13, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)

    # 우축: 유저 비중
    ax1b = ax1.twinx()
    ax1b.plot(GROUP_ORDER, group_pct.values, 'ko-', linewidth=2.5, markersize=10, zorder=5)
    for i, (x, y) in enumerate(zip(GROUP_ORDER, group_pct.values)):
        ax1b.annotate(f'{y:.0f}%', xy=(x, y), xytext=(0, 10),
                      textcoords='offset points', ha='center', fontsize=11,
                      fontweight='bold', color='black')
    ax1b.set_ylabel('유저 비중 (%)', fontsize=11)
    ax1b.set_ylim(0, 70)

    # 핵심 메시지
    ax1.text(0.02, 0.95,
             f'1000조이상: ARPU 최고 but 유저 {group_pct["1000조이상"]:.0f}%\n'
             f'10~100조+100~1000조: 유저 {core_pct:.0f}% (핵심 유저층)',
             transform=ax1.transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    # ═══════════════════════════════════════════════════
    # (우상) ② 그룹별 이탈 유저 수 — 89%의 핵심 유저층이 대규모 이탈
    # ═══════════════════════════════════════════════════
    ax2 = axes[0, 1]

    bars2 = ax2.bar(GROUP_ORDER, churned_count.values, color=group_colors,
                    edgecolor='white', linewidth=1.5)
    # 10~100조, 100~1000조 강조 테두리
    bars2[1].set_edgecolor(core_color)
    bars2[1].set_linewidth(3)
    bars2[2].set_edgecolor(core_color)
    bars2[2].set_linewidth(3)

    for bar, cnt, rate in zip(bars2, churned_count.values, churn_rate.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{int(cnt):,}명\n(이탈률 {rate:.1f}%)',
                 ha='center', fontsize=9, fontweight='bold')

    ax2.set_ylabel('이탈 유저 수 (명)', fontsize=11)
    ax2.set_title('② 패키지 출시 → 핵심 유저층(89%)에서 대규모 이탈', fontsize=13, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    # 핵심 유저층 합계 강조
    core_churned = churned_count['10~100조'] + churned_count['100~1000조']
    total_churned = churned_count.sum()
    core_churn_pct = core_churned / total_churned * 100
    ax2.annotate(f'10~100조 + 100~1000조\n= 전체 이탈의 {core_churn_pct:.0f}%\n'
                 f'({int(core_churned):,}명 / {int(total_churned):,}명)',
                 xy=(1.5, max(churned_count.values) * 0.7),
                 fontsize=11, color=core_color, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff0f0', alpha=0.95))

    # ═══════════════════════════════════════════════════
    # (좌하) ③ 이탈 손실 분해 — 이탈자 수 × 실제 월 요금
    # ═══════════════════════════════════════════════════
    ax3 = axes[1, 0]

    # 스택 바: 이탈자수 × 평균월요금 = 총 손실
    loss_pct = group_loss / group_loss.sum() * 100
    bars3 = ax3.bar(GROUP_ORDER, group_loss.values, color=group_colors,
                    edgecolor='white', linewidth=1.5)
    bars3[1].set_edgecolor(core_color)
    bars3[1].set_linewidth(3)
    bars3[2].set_edgecolor(core_color)
    bars3[2].set_linewidth(3)

    for bar, val, pct in zip(bars3, group_loss.values, loss_pct.values):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}억 ({pct:.0f}%)', ha='center', fontsize=10, fontweight='bold')

    ax3.set_ylabel('이탈 손실 (억 원)', fontsize=11)
    ax3.set_title('③ 이탈 손실 분해 — 이탈자수 × (월 멤버십 + 월 과금) × 8개월', fontsize=13, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)

    # 핵심 유저층 합계
    core_loss = group_loss['10~100조'] + group_loss['100~1000조']
    core_loss_pct = core_loss / group_loss.sum() * 100
    ax3.text(0.95, 0.95,
             f'핵심 유저층(89%) 이탈 손실:\n'
             f'{core_loss:.1f}억 = 전체 손실의 {core_loss_pct:.0f}%\n\n'
             f'1000조이상: 1인당 손실은 최대이지만\n'
             f'이탈자 수가 적어 총 손실 비중은 낮음',
             transform=ax3.transAxes, fontsize=9, va='top', ha='right',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    # ═══════════════════════════════════════════════════
    # (우하) ④ 영업이익 관점 종합 — 패키지 매출 vs 핵심 유저층 이탈 손실
    # ═══════════════════════════════════════════════════
    ax4 = axes[1, 1]

    # 비교 항목
    total_pkg_rev = pkg_revenue.sum() / 1e8
    labels = ['패키지 매출\n(전 그룹)', f'핵심 유저층\n이탈 손실\n(10~100조+100~1000조)',
              '전체\n이탈 손실']
    values = [total_pkg_rev, core_loss, group_loss.sum()]
    bar_colors = ['#2ecc71', core_color, '#c0392b']

    bars4 = ax4.bar(labels, values, color=bar_colors, width=0.6,
                    edgecolor='white', linewidth=1.5)

    for bar, val in zip(bars4, values):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}억', ha='center', fontsize=13, fontweight='bold')

    # 손익 차이 화살표
    net_core = total_pkg_rev - core_loss
    ax4.annotate(f'핵심 유저층만으로도\n순손실 {abs(net_core):.1f}억',
                 xy=(1, core_loss * 0.5),
                 xytext=(1.8, core_loss * 0.6),
                 fontsize=11, color=core_color, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=core_color, lw=2.5),
                 bbox=dict(boxstyle='round', facecolor='#fff0f0', alpha=0.95))

    ax4.set_ylabel('금액 (억 원)', fontsize=11)
    ax4.set_title('④ 영업이익 관점: 패키지 매출로 이탈 손실을 메울 수 없다', fontsize=13, fontweight='bold')
    ax4.grid(axis='y', alpha=0.3)

    # 최종 결론 박스
    ax4.text(0.02, 0.95,
             f'패키지 매출 {total_pkg_rev:.1f}억\n'
             f'  < 핵심 유저층(89%) 이탈 손실 {core_loss:.1f}억\n'
             f'  < 전체 이탈 손실 {group_loss.sum():.1f}억\n\n'
             f'→ 패키지가 89%의 유저를 이탈시키면\n'
             f'   영업이익이 구조적으로 악화됨',
             transform=ax4.transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='#ffe6e6', alpha=0.95))

    fig.suptitle('패키지 출시가 게임사 영업이익에 미치는 영향\n'
                 f'10~100조 + 100~1000조 = 전체 유저의 {core_pct:.0f}% → 이탈 시 영업이익 구조적 손실',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()

    save_fig(fig, 'fig_12b_why_mid_group_matters.png')


# ============================================================================
# Fig 12c: 구단가치 그룹별 평균 멤버십 티어 및 평균 과금액
# ============================================================================
def fig_12c_group_tier_spending(user_profile):
    """
    구단가치 그룹별 과금 구조 시각화:
      (좌) 그룹별 멤버십 티어 분포 — 100% 스택 바
      (우) 그룹별 평균 월 과금액 + 평균 멤버십 요금 — 스택 바
    """
    print("\n[Fig 12c] 그룹별 평균 멤버십 티어 및 과금 구조")

    TIER_ORDER = ['Bronze', 'Silver', 'Gold', 'Platinum', 'Diamond']
    TIER_COLORS = {
        'Bronze':   '#cd7f32',  # 브론즈 갈색
        'Silver':   '#c0c0c0',  # 실버 회색
        'Gold':     '#ffd700',  # 골드 노란색
        'Platinum': '#4169e1',  # 플래티넘 파란색
        'Diamond':  '#b9f2ff',  # 다이아몬드 하늘색
    }

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

    # ── (좌) 멤버십 티어 100% 스택 바 ──
    tier_pct = pd.crosstab(
        user_profile['club_value_group'], user_profile['membership_tier'],
        normalize='index') * 100
    tier_pct = tier_pct.reindex(GROUP_ORDER)[TIER_ORDER]

    bottom = np.zeros(len(GROUP_ORDER))
    for tier in TIER_ORDER:
        vals = tier_pct[tier].values
        bars = ax1.bar(GROUP_ORDER, vals, bottom=bottom, label=tier,
                       color=TIER_COLORS[tier], edgecolor='white', linewidth=0.5)
        # 10% 이상인 구간에만 레이블 표시
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v >= 8:
                ax1.text(i, b + v/2, f'{v:.0f}%', ha='center', va='center',
                         fontsize=9, fontweight='bold')
        bottom += vals

    ax1.set_ylabel('티어 비중 (%)', fontsize=12)
    ax1.set_title('그룹별 멤버십 티어 분포', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=10, title='멤버십 등급')
    ax1.set_ylim(0, 105)
    ax1.grid(axis='y', alpha=0.2)

    # 각 그룹의 평균 티어 텍스트
    tier_numeric = {'Bronze': 1, 'Silver': 2, 'Gold': 3, 'Platinum': 4, 'Diamond': 5}
    for i, g in enumerate(GROUP_ORDER):
        sub = user_profile[user_profile['club_value_group'] == g]
        avg_tier_num = sub['membership_tier'].map(tier_numeric).mean()
        # 숫자 → 티어명 (가장 가까운 티어)
        closest_tier = min(tier_numeric, key=lambda t: abs(tier_numeric[t] - avg_tier_num))
        ax1.text(i, 102, f'평균: {closest_tier}({avg_tier_num:.1f})',
                 ha='center', fontsize=9, fontweight='bold', color='#333')

    # ── (우) 평균 월 과금액 + 멤버십 스택 바 ──
    avg_membership = user_profile.groupby('club_value_group')['monthly_membership_fee'].mean()
    avg_spending = user_profile.groupby('club_value_group')['monthly_avg_spending'].mean()
    avg_membership = avg_membership.reindex(GROUP_ORDER)
    avg_spending = avg_spending.reindex(GROUP_ORDER)

    # 만원 단위 변환
    mem_val = avg_membership.values / 1e4
    spend_val = avg_spending.values / 1e4

    bars_mem = ax2.bar(GROUP_ORDER, mem_val, label='월 멤버십 요금',
                       color='#3498db', edgecolor='white', linewidth=1)
    bars_spend = ax2.bar(GROUP_ORDER, spend_val, bottom=mem_val,
                         label='월 평균 과금액', color='#e67e22',
                         edgecolor='white', linewidth=1)

    # 멤버십 레이블
    for i, (mv, sv) in enumerate(zip(mem_val, spend_val)):
        ax2.text(i, mv/2, f'{mv:.1f}만', ha='center', va='center',
                 fontsize=9, fontweight='bold', color='white')
        # 과금 레이블
        if sv >= 1:
            ax2.text(i, mv + sv/2, f'{sv:.1f}만', ha='center', va='center',
                     fontsize=9, fontweight='bold', color='white')
        # 합계 레이블 (위에)
        total = mv + sv
        ax2.text(i, total + max(spend_val) * 0.03,
                 f'합계 {total:.1f}만원', ha='center', fontsize=10,
                 fontweight='bold', color='#333')

    ax2.set_ylabel('월 기대수익 (만원)', fontsize=12)
    ax2.set_title('그룹별 월 평균 멤버십 + 과금액', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(axis='y', alpha=0.3)

    # 과금 배율 주석
    ratio_whale = (avg_membership['1000조이상'] + avg_spending['1000조이상']) / \
                  (avg_membership['10~100조'] + avg_spending['10~100조'])
    ax2.text(0.98, 0.95,
             f'1000조이상 유저 1명의 월 기대수익은\n'
             f'10~100조 유저의 약 {ratio_whale:.0f}배',
             transform=ax2.transAxes, fontsize=10, va='top', ha='right',
             fontweight='bold', color='#c0392b',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    fig.suptitle('구단가치 그룹별 과금 구조 분석 — 멤버십 티어 × 월 과금액',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()

    # 그룹별 상세 출력
    for g in GROUP_ORDER:
        sub = user_profile[user_profile['club_value_group'] == g]
        print(f"  {g}: 평균 멤버십 {avg_membership[g]:,.0f}원 + "
              f"평균 과금 {avg_spending[g]:,.0f}원 = "
              f"합계 {avg_membership[g]+avg_spending[g]:,.0f}원/월")

    save_fig(fig, 'fig_12c_group_tier_spending.png')


# ============================================================================
# ★ 가설 근거 시각화 (ML 피처 선택 근거)
# ============================================================================

# Fig A1: 유저 OVR ↔ 패키지 OVR 겹침 → 구단가치 하락 관계
def fig_a1_ovr_overlap_decline(user_profile, dcv):
    """
    가설1 근거: 패키지에서 높은 확률(130,133)로 등장하는 OVR의 선수를
    보유한 유저일수록 구단가치 하락이 크다는 것을 시각화
    X축: 유저 평균 OVR, Y축: 구단가치 하락률(%), 색상: 그룹
    """
    print("\n[Fig A1] OVR 겹침도 vs 구단가치 하락률")

    # dcv에서 유저별 구단가치 하락률 계산
    first_date = dcv['date'].min()
    last_date = dcv['date'].max()
    cv_first = dcv[dcv['date'] == first_date][['user_id', 'club_value']].copy()
    cv_last = dcv[dcv['date'] == last_date][['user_id', 'club_value']].copy()
    cv_first.columns = ['user_id', 'cv_start']
    cv_last.columns = ['user_id', 'cv_end']

    decline = cv_first.merge(cv_last, on='user_id')
    decline['decline_pct'] = (decline['cv_start'] - decline['cv_end']) / decline['cv_start'] * 100
    decline = decline.merge(user_profile[['user_id', 'avg_ovr', 'club_value_group']], on='user_id')

    fig, ax = plt.subplots(figsize=(14, 8))

    for g in GROUP_ORDER:
        gdata = decline[decline['club_value_group'] == g]
        ax.scatter(gdata['avg_ovr'], gdata['decline_pct'],
                   c=COLORS[g], alpha=0.4, s=20, label=g)

    # 패키지 높은 확률 OVR 영역 강조
    ax.axvspan(129.5, 134.5, alpha=0.1, color='red',
               label='패키지 높은확률 OVR (130~134)')
    ax.axvline(133, color='red', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.axvline(130, color='red', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.text(131.5, ax.get_ylim()[1] * 0.9, 'OVR 130\n(19%)', color='red',
            ha='center', fontsize=9, fontweight='bold')
    ax.text(133.5, ax.get_ylim()[1] * 0.85, 'OVR 133\n(40%)', color='red',
            ha='center', fontsize=9, fontweight='bold')

    ax.set_xlabel('유저 평균 OVR', fontsize=12)
    ax.set_ylabel('구단가치 하락률 (%)', fontsize=12)
    ax.set_title('가설1 근거: 유저 보유 OVR이 패키지 높은확률 OVR과 겹칠수록\n구단가치 하락이 크다',
                 fontsize=14)
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'fig_a1_ovr_overlap_decline.png')


# Fig A2: 패키지 출시 전후 그룹별 구단가치 변화 (Facet)
def fig_a2_pkg_before_after(dcv):
    """
    가설2 근거: 각 그룹별로 패키지 출시 전후 구단가치 인덱스가
    어떻게 변하는지 Facet으로 비교
    설계서 멘토 조언: "그룹별로 패키지 출시 시점에 따라 하락률과 이탈률을 같이 걸어서 비교"
    """
    print("\n[Fig A2] 패키지 전후 구단가치 변화 (그룹별 Facet)")

    # 1차 패키지(1/18) 기준 전후 20일
    pkg_date = pd.Timestamp('2025-01-18')
    window = 20

    nearby = dcv[(dcv['date'] >= pkg_date - pd.Timedelta(days=window)) &
                 (dcv['date'] <= pkg_date + pd.Timedelta(days=window))].copy()
    nearby['days_from_pkg'] = (nearby['date'] - pkg_date).dt.days

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)

    for idx, (g, ax) in enumerate(zip(GROUP_ORDER, axes.flatten())):
        gdata = nearby[nearby['club_value_group'] == g]
        daily_avg = gdata.groupby('days_from_pkg')['club_value_index'].mean()

        ax.plot(daily_avg.index, daily_avg.values,
                color=COLORS[g], linewidth=2.5, marker='o', markersize=3)
        ax.axvline(0, color='red', linestyle='--', linewidth=2, alpha=0.7)
        ax.axhline(100, color='gray', linestyle=':', alpha=0.4)

        # 전후 하락률 표시
        pre_avg = daily_avg[daily_avg.index < 0].mean()
        post_avg = daily_avg[daily_avg.index > 0].mean()
        drop = pre_avg - post_avg
        ax.text(0.05, 0.1, f'하락: {drop:.1f}pt',
                transform=ax.transAxes, fontsize=12, fontweight='bold',
                color='red', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

        ax.set_title(f'{g}', fontsize=13, fontweight='bold', color=COLORS[g])
        ax.set_xlabel('패키지 출시 후 일수', fontsize=10)
        ax.grid(True, alpha=0.3)

    axes[0][0].set_ylabel('구단가치 인덱스', fontsize=11)
    axes[1][0].set_ylabel('구단가치 인덱스', fontsize=11)

    fig.suptitle('가설2 근거: 1차 패키지(1/18) 출시 전후 그룹별 구단가치 변화\n→ 10~100조 그룹 하락이 가장 큼',
                 fontsize=15, fontweight='bold', y=1.02)
    fig.tight_layout()
    save_fig(fig, 'fig_a2_pkg_before_after.png')


# Fig A3: 인과 체인 요약 — 하락률 vs 이탈률 (그룹별 + OVR 겹침도 버블)
def fig_a3_causal_chain(user_profile, dcv, churn_data, trade_market):
    """
    가설 전체 인과 체인:
    패키지OVR확률 → 선수가격하락 → 구단가치하락 → 이탈
    이 체인을 하나의 차트에서 보여줌
    X: OVR가격하락률, Y: 구단가치하락률, 버블크기: 이탈률, 색상: 그룹
    """
    print("\n[Fig A3] 인과 체인 요약")

    # 그룹별 평균 OVR
    group_ovr = user_profile.groupby('club_value_group')['avg_ovr'].mean()

    # OVR별 가격 하락률 — 패키지 전(~1/17) vs 패키지 후(3/21~) 비교
    trade_market['trade_date'] = pd.to_datetime(trade_market['trade_date'])
    pre_pkg = trade_market[trade_market['trade_date'] < '2025-01-18']
    post_pkg = trade_market[trade_market['trade_date'] > '2025-03-20']
    pre_prices = pre_pkg.groupby('ovr')['price_trade'].mean()
    post_prices = post_pkg.groupby('ovr')['price_trade'].mean()
    ovr_drop = ((pre_prices - post_prices) / pre_prices * 100)

    # 그룹별 "보유 OVR대의 가중 평균 가격 하락률" 계산
    # 각 유저의 avg_ovr를 반올림하여 해당 OVR의 가격 하락률 매핑
    user_profile['ovr_rounded'] = user_profile['avg_ovr'].round().astype(int)
    user_profile['ovr_price_drop'] = user_profile['ovr_rounded'].map(ovr_drop).fillna(0)
    group_price_drop = user_profile.groupby('club_value_group')['ovr_price_drop'].mean()
    group_price_drop = group_price_drop.reindex(GROUP_ORDER)

    # 그룹별 구단가치 하락률
    first_date = dcv['date'].min()
    last_date = dcv['date'].max()
    cv_first = dcv[dcv['date'] == first_date].groupby('club_value_group')['club_value'].mean()
    cv_last = dcv[dcv['date'] == last_date].groupby('club_value_group')['club_value'].mean()
    cv_decline = ((cv_first - cv_last) / cv_first * 100).reindex(GROUP_ORDER)

    # 그룹별 이탈률
    churn_rate = churn_data.groupby('club_value_group')['is_churned'].mean() * 100
    churn_rate = churn_rate.reindex(GROUP_ORDER)

    fig, ax = plt.subplots(figsize=(12, 8))

    for g in GROUP_ORDER:
        pd_val = group_price_drop[g]
        cv_val = cv_decline[g]
        cr_val = churn_rate[g]

        ax.scatter(pd_val, cv_val,
                   s=cr_val * 40,  # 버블 크기 = 이탈률
                   c=COLORS[g], edgecolors='black', linewidths=1.5, alpha=0.85, zorder=5)
        ax.annotate(f'{g}\nOVR가격하락: {pd_val:.1f}%\n구단가치하락: {cv_val:.1f}%\n이탈률: {cr_val:.1f}%',
                    xy=(pd_val, cv_val),
                    xytext=(15, 15), textcoords='offset points',
                    fontsize=9, fontweight='bold', color=COLORS[g],
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor=COLORS[g], alpha=0.9),
                    arrowprops=dict(arrowstyle='->', color=COLORS[g], lw=1.2))

    # 추세선 + 상관계수 (4개 그룹 데이터 포인트)
    x_vals = [group_price_drop[g] for g in GROUP_ORDER]
    y_vals = [cv_decline[g] for g in GROUP_ORDER]
    z = np.polyfit(x_vals, y_vals, 1)
    p = np.poly1d(z)
    # 피어슨 상관계수 계산
    corr_r, corr_p = stats.pearsonr(x_vals, y_vals)
    x_line = np.linspace(min(x_vals) - 0.3, max(x_vals) + 0.3, 50)
    ax.plot(x_line, p(x_line), '--', color='gray', alpha=0.5, linewidth=1.5,
            label=f'추세선 (r={corr_r:.3f}, p={corr_p:.3f})')

    ax.set_xlabel('보유 OVR대 선수 가격 하락률 (%)\n← 패키지 OVR 등장확률이 높을수록 가격 하락',
                  fontsize=12)
    ax.set_ylabel('구단가치 하락률 (%)\n← OVR 가격 하락이 구단가치에 직접 반영',
                  fontsize=12)
    ax.set_title('가설 인과 체인: 패키지OVR확률 → 선수가격하락 → 구단가치하락 → 이탈\n(버블 크기 = 이탈률)',
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10, loc='lower right')

    # 범례 설명
    ax.text(0.02, 0.98, '● 버블이 클수록 이탈률 높음\n→ 가격 하락 노출이 큰 그룹일수록\n   구단가치 하락·이탈이 심함',
            transform=ax.transAxes, fontsize=10, va='top', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    save_fig(fig, 'fig_a3_causal_chain.png')


# ============================================================================
# 메인 실행
# ============================================================================
def main():
    # 데이터 로드
    user_profile, login_logs, package_purchase, trade_market, dcv = load_all_data()

    # 이탈 지표 계산
    churn_data = compute_churn(user_profile, login_logs)
    churn_rate = churn_data.groupby('club_value_group')['is_churned'].mean() * 100
    print("\n[이탈률 검증]")
    for g in GROUP_ORDER:
        print(f"  {g}: {churn_rate.get(g, 0):.1f}%")

    # ── EDA 차트 생성 (PPT 사용 차트만) ──
    fig_01_kmeans_elbow(user_profile)
    fig_02_group_distribution(user_profile)
    fig_02_ovr_violin(user_profile, dcv)
    fig_03_package_ovr_bubble(package_purchase)
    fig_05_package_sales(package_purchase)
    fig_07_club_value_index(dcv)
    fig_08_ovr_price_trend(trade_market)
    fig_09_did_analysis(dcv, churn_data)
    fig_10_churn_by_group(churn_data)
    fig_11_sensitivity_analysis(dcv, churn_data)
    fig_12_revenue_vs_loss(user_profile, churn_data, package_purchase)
    fig_12b_why_mid_group_matters(user_profile, churn_data, package_purchase)
    fig_12c_group_tier_spending(user_profile)

    # ── 가설 근거 시각화 (ML 피처 선택의 근거) ──
    print("\n" + "=" * 70)
    print("가설 근거 시각화 — ML 피처 선택 전 인과관계 증명")
    print("=" * 70)
    fig_a1_ovr_overlap_decline(user_profile, dcv)
    fig_a2_pkg_before_after(dcv)
    fig_a3_causal_chain(user_profile, dcv, churn_data, trade_market)

    print("\n" + "=" * 70)
    print(f"전체 16개 차트 생성 완료! (EDA 13 + 가설근거 3)")
    print(f"저장 경로: {OUTPUT_PATH}")
    print("=" * 70)


main()

In [ ]:
"""
=============================================================================
FC Online 4 데이터 분석 프로젝트 - Stage 3: ML 이탈 예측 모델
=============================================================================
모델: Random Forest + Logistic Regression
목적: 패키지 출시 시 이탈 유저를 사전 예측하는 모델 구축

■ 피처 선택 근거 (02_eda_visualization.py 가설 근거 시각화에서 증명)
  ─────────────────────────────────────────────────────
  [fig_a1] 유저 보유 OVR이 패키지 높은확률 OVR(130~134)과 겹칠수록
           구단가치 하락이 크다 → ovr_pkg_overlap, is_pkg_ovr_range
  [fig_a2] 패키지 출시 전후 10~100조 그룹의 구단가치 하락이 가장 큼
           → pkg1_cv_change, pkg2_cv_change, pkg_cumulative_shock
  [fig_a3] 인과 체인: OVR가격하락 → 구단가치하락 → 이탈
           (추세선 기울기 양의 상관) → ovr_price_exposure

  위 EDA 시각화에서 증명된 상관관계 및 연관성을 바탕으로 아래 핵심 피처를 설계:
  ★ ovr_pkg_overlap     : 유저 OVR↔패키지 OVR 겹침도 (가설1 근거)
  ★ is_pkg_ovr_range    : 패키지 높은확률 OVR 범위 해당 여부 (가설1 근거)
  ★ pkg1/2_cv_change    : 패키지 출시 전후 구단가치 변화율 (가설2 근거)
  ★ pkg_cumulative_shock: 1차+2차 패키지 누적 충격 (가설2 근거)
  ★ ovr_price_exposure  : 보유 OVR대 가격 하락 노출도 (인과체인 근거)

■ 추가 피처 (일반적 이탈 예측 지표)
  - 유저 프로필: club_value, avg_ovr, spendig, membership_tier
  - 접속 패턴: 접속일수, 세션시간, 접속감소율, 마지막주접속
  - 구매 행동: 패키지 구매횟수, 금액, 다양성
  - 구단가치 변동: 인덱스, 최대하락률, z-score 충격횟수
  - 거래 활동: 거래 참여 횟수

■ 데이터 누출 방지
  관측 기간 (피처): 2025-01-01 ~ 2025-03-01
  결과 기간 (타겟): 2025-03-01 ~ 2025-04-30

타겟: is_churned (결과 기간 30일 미접속)
=============================================================================
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)

# 폰트 설정
sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings('ignore')
setup_korean_font()

# ============================================================================
# 경로 설정
# ============================================================================
BASE_DIR = PROJECT_ROOT
DATA_PATH = os.path.join(BASE_DIR, 'data')
OUTPUT_PATH = os.path.join(BASE_DIR, 'outputs', 'figures')
MODEL_PATH = os.path.join(BASE_DIR, 'outputs', 'models')
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

# 핵심 파라미터
DATA_END = pd.Timestamp('2025-04-30')
CHURN_DAYS = 30
SEED = 42

print("=" * 70)
print("FC Online 4 - Stage 3: ML 이탈 예측 모델")
print("=" * 70)


# ============================================================================
# 1. 데이터 로드
# ============================================================================
def load_data():
    """5개 테이블 로드"""
    print("\n[1] 데이터 로드")
    up = pd.read_csv(os.path.join(DATA_PATH, 'user_profile.csv'))
    ll = pd.read_csv(os.path.join(DATA_PATH, 'login_logs.csv'))
    pp = pd.read_csv(os.path.join(DATA_PATH, 'package_purchase.csv'))
    tm = pd.read_csv(os.path.join(DATA_PATH, 'trade_market.csv'))
    dcv = pd.read_csv(os.path.join(DATA_PATH, 'daily_club_value.csv'))

    ll['login_date'] = pd.to_datetime(ll['login_date'])
    pp['purchase_date'] = pd.to_datetime(pp['purchase_date'])
    tm['trade_date'] = pd.to_datetime(tm['trade_date'])
    dcv['date'] = pd.to_datetime(dcv['date'])

    for name, df in [('user_profile', up), ('login_logs', ll),
                     ('package_purchase', pp), ('trade_market', tm),
                     ('daily_club_value', dcv)]:
        print(f"  {name}: {len(df):>10,}행")
    return up, ll, pp, tm, dcv


# ============================================================================
# 2. 피처 엔지니어링
# ============================================================================
def engineer_features(up, ll, pp, tm, dcv):
    """
    유저별 피처 생성 — 시간 분할(Temporal Split)로 데이터 누출 방지

    관측 기간 (피처 계산): 2025-01-01 ~ 2025-03-01 (60일)
    결과 기간 (이탈 판정): 2025-03-01 ~ 2025-04-30 (60일)
    이탈 정의: 결과 기간 중 마지막 접속 후 30일 미접속
    """
    print("\n[2] 피처 엔지니어링 (시간 분할 — 데이터 누출 방지)")

    # 시간 분할 기준
    OBS_END = pd.Timestamp('2025-03-01')      # 관측 기간 종료
    OUTCOME_START = pd.Timestamp('2025-03-01') # 결과 기간 시작

    # ── 타겟 변수: 결과 기간 이탈 여부 ──
    # 결과 기간(3/1~4/30) 동안 접속 기록
    outcome_logs = ll[ll['login_date'] >= OUTCOME_START]
    last_login_outcome = outcome_logs.groupby('user_id')['login_date'].max().reset_index()
    last_login_outcome.columns = ['user_id', 'last_login_outcome']
    last_login_outcome['days_inactive_outcome'] = (DATA_END - last_login_outcome['last_login_outcome']).dt.days
    last_login_outcome['is_churned'] = (last_login_outcome['days_inactive_outcome'] > CHURN_DAYS).astype(int)

    # 결과 기간에 아예 접속 안 한 유저 → 이탈
    all_users = set(up['user_id'])
    outcome_users = set(last_login_outcome['user_id'])
    never_logged = all_users - outcome_users

    never_df = pd.DataFrame({
        'user_id': list(never_logged),
        'last_login_outcome': pd.NaT,
        'days_inactive_outcome': 999,
        'is_churned': 1
    })
    last_login_outcome = pd.concat([last_login_outcome, never_df], ignore_index=True)

    # ── 관측 기간 로그만 사용 (데이터 누출 방지) ──
    obs_logs = ll[ll['login_date'] < OBS_END]
    obs_purchases = pp[pp['purchase_date'] < OBS_END]
    obs_trades = tm[tm['trade_date'] < OBS_END]
    obs_dcv = dcv[dcv['date'] < OBS_END]

    # ── 기본 프로필 피처 ──
    features = up[['user_id', 'club_value_group', 'club_value',
                    'avg_ovr', 'spendig', 'membership_tier']].copy()

    tier_map = {'Bronze': 1, 'Silver': 2, 'Gold': 3, 'Platinum': 4, 'Diamond': 5}
    features['membership_rank'] = features['membership_tier'].map(tier_map)
    features['club_value_jo'] = features['club_value'] / 1e12

    print("  [2-1] 접속 패턴 피처 생성 중 (관측 기간만 사용)...")

    # ── 접속 패턴 피처 (관측 기간에서만) ──
    login_days = obs_logs.groupby('user_id')['login_date'].nunique().reset_index()
    login_days.columns = ['user_id', 'obs_login_days']

    avg_session = obs_logs.groupby('user_id')['session_duration_min'].mean().reset_index()
    avg_session.columns = ['user_id', 'obs_avg_session']

    # 관측 기간 전반부 (1/1~1/31) vs 후반부 (2/1~2/28) 접속 비교
    early_logs = obs_logs[obs_logs['login_date'] < pd.Timestamp('2025-02-01')]
    late_logs = obs_logs[obs_logs['login_date'] >= pd.Timestamp('2025-02-01')]

    early_cnt = early_logs.groupby('user_id')['login_date'].nunique().reset_index()
    early_cnt.columns = ['user_id', 'early_logins']

    late_cnt = late_logs.groupby('user_id')['login_date'].nunique().reset_index()
    late_cnt.columns = ['user_id', 'late_logins']

    # 관측기간 마지막 주(2/22~2/28) 접속 빈도 (이탈 전조 파악)
    last_week = obs_logs[obs_logs['login_date'] >= pd.Timestamp('2025-02-22')]
    last_week_cnt = last_week.groupby('user_id')['login_date'].nunique().reset_index()
    last_week_cnt.columns = ['user_id', 'last_week_logins']

    print("  [2-2] 구매 행동 피처 생성 중 (관측 기간만)...")

    # ── 구매 행동 (관측 기간) ──
    pkg_count = obs_purchases.groupby('user_id').size().reset_index(name='obs_pkg_count')
    pkg_amount = obs_purchases.groupby('user_id')['amount'].sum().reset_index()
    pkg_amount.columns = ['user_id', 'obs_pkg_amount']
    pkg_variety = obs_purchases.groupby('user_id')['package_id'].nunique().reset_index()
    pkg_variety.columns = ['user_id', 'obs_pkg_variety']
    pkg_ovr = obs_purchases.groupby('user_id')['ovr'].mean().reset_index()
    pkg_ovr.columns = ['user_id', 'obs_avg_pkg_ovr']

    print("  [2-3] 설계서 가설 핵심 피처 생성 중...")

    # =========================================================================
    # ★ 설계서 핵심 가설 기반 피처 (인과 체인 반영)
    # =========================================================================
    # 가설1: 패키지 → 높은확률 OVR 선수 공급 증가 → 해당 OVR 가격 하락
    # 가설2: 유저 보유 OVR이 패키지 높은확률 OVR과 겹칠수록 구단가치 하락
    # 가설3: 구단가치 하락이 큰 그룹(10~100조)이 이탈에 가장 민감

    # ── (A) 유저 보유 OVR과 패키지 높은확률 OVR 겹침도 ──
    # 패키지에서 높은 확률 OVR: 130(19%), 133(40%) → 이 OVR대 선수 보유시 직격탄
    # 유저 avg_ovr이 130~134 범위에 가까울수록 패키지 영향 직접 받음
    PKG_HIGH_PROB_OVRS = [130, 131, 132, 133, 134]  # 패키지 확률 높은 OVR 대역
    PKG_CENTER_OVR = 132.0  # 패키지 확률 분포의 중심

    features['ovr_pkg_distance'] = abs(features['avg_ovr'] - PKG_CENTER_OVR)
    # 거리가 가까울수록 겹침도가 높음 → 역수로 변환 (최대 1.0)
    features['ovr_pkg_overlap'] = 1.0 / (1.0 + features['ovr_pkg_distance'])

    # 유저의 OVR이 패키지 높은확률 범위(130~134)에 속하는지 여부
    features['is_pkg_ovr_range'] = features['avg_ovr'].between(129.5, 134.5).astype(int)

    # ── (B) 패키지 출시 전후 구단가치 변화 (1차: 1/18, 2차: 2/20) ──
    # 각 패키지 출시일 기준 전 7일 vs 후 7일 구단가치 비교
    PKG1_DATE = pd.Timestamp('2025-01-18')
    PKG2_DATE = pd.Timestamp('2025-02-20')

    for pkg_name, pkg_date in [('pkg1', PKG1_DATE), ('pkg2', PKG2_DATE)]:
        pre_start = pkg_date - pd.Timedelta(days=7)
        post_end = pkg_date + pd.Timedelta(days=7)

        # 관측 기간 내인지 확인
        if post_end <= OBS_END:
            pre_cv = obs_dcv[(obs_dcv['date'] >= pre_start) & (obs_dcv['date'] < pkg_date)]
            post_cv = obs_dcv[(obs_dcv['date'] >= pkg_date) & (obs_dcv['date'] <= post_end)]

            pre_avg = pre_cv.groupby('user_id')['club_value_index'].mean().reset_index()
            pre_avg.columns = ['user_id', f'{pkg_name}_pre_idx']

            post_avg = post_cv.groupby('user_id')['club_value_index'].mean().reset_index()
            post_avg.columns = ['user_id', f'{pkg_name}_post_idx']

            features = features.merge(pre_avg, on='user_id', how='left')
            features = features.merge(post_avg, on='user_id', how='left')

            # 패키지 전후 변화율 (음수 = 하락)
            features[f'{pkg_name}_cv_change'] = (
                (features[f'{pkg_name}_post_idx'] - features[f'{pkg_name}_pre_idx'])
                / features[f'{pkg_name}_pre_idx'].replace(0, 1) * 100
            )
            features[f'{pkg_name}_cv_change'] = features[f'{pkg_name}_cv_change'].fillna(0)

    # 1차+2차 패키지 누적 충격 (음수값 합산)
    features['pkg_cumulative_shock'] = (
        features.get('pkg1_cv_change', pd.Series(0, index=features.index)).clip(upper=0) +
        features.get('pkg2_cv_change', pd.Series(0, index=features.index)).clip(upper=0)
    )

    # ── (C) 보유 OVR대 가격 하락 노출도 (trade_market 기반) ──
    # 유저의 avg_ovr에 해당하는 OVR의 관측기간 내 가격 하락률 매핑
    # 각 OVR별 전체 기간 가격 변화 계산
    ovr_first_price = obs_trades.groupby('ovr').apply(
        lambda x: x.nsmallest(max(1, len(x)//10), 'trade_date')['price_trade'].mean()
    ).reset_index(name='first_price')

    ovr_last_price = obs_trades.groupby('ovr').apply(
        lambda x: x.nlargest(max(1, len(x)//10), 'trade_date')['price_trade'].mean()
    ).reset_index(name='last_price')

    ovr_price_change = ovr_first_price.merge(ovr_last_price, on='ovr')
    ovr_price_change['price_drop_pct'] = (
        (ovr_price_change['first_price'] - ovr_price_change['last_price'])
        / ovr_price_change['first_price'] * 100
    )

    # 유저의 avg_ovr을 반올림하여 가장 가까운 OVR의 가격 하락률 매핑
    features['ovr_rounded'] = features['avg_ovr'].round().astype(int)
    price_drop_map = dict(zip(ovr_price_change['ovr'], ovr_price_change['price_drop_pct']))
    features['ovr_price_exposure'] = features['ovr_rounded'].map(price_drop_map).fillna(0)
    features.drop('ovr_rounded', axis=1, inplace=True)

    print("    → ovr_pkg_overlap: 유저 OVR↔패키지 OVR 겹침도")
    print("    → is_pkg_ovr_range: 패키지 높은확률 OVR 범위(130~134) 해당 여부")
    print("    → pkg1/2_cv_change: 패키지 출시 전후 구단가치 변화율")
    print("    → pkg_cumulative_shock: 누적 패키지 충격")
    print("    → ovr_price_exposure: 보유 OVR 가격 하락 노출도")

    print("\n  [2-4] 구단가치 변동 피처 생성 중 (관측 기간만)...")

    # ── 구단가치 변동 (관측 기간) ──
    obs_dcv_last = obs_dcv.groupby('user_id').last().reset_index()
    cv_obs_index = obs_dcv_last[['user_id', 'club_value_index']].copy()
    cv_obs_index.columns = ['user_id', 'cv_obs_index']

    cv_max_drop = obs_dcv.groupby('user_id')['daily_change_rate'].min().reset_index()
    cv_max_drop.columns = ['user_id', 'cv_max_daily_drop']

    shock_count = obs_dcv[obs_dcv['z_score'] <= -2].groupby('user_id').size().reset_index(name='shock_count')

    cv_avg_change = obs_dcv.groupby('user_id')['daily_change_rate'].mean().reset_index()
    cv_avg_change.columns = ['user_id', 'cv_avg_change_rate']

    # 구단가치 하락폭 (1차+2차 패키지 효과만 반영)
    first_date = obs_dcv['date'].min()
    cv_first = obs_dcv[obs_dcv['date'] == first_date][['user_id', 'club_value']].copy()
    cv_first.columns = ['user_id', 'cv_first']
    cv_last = obs_dcv_last[['user_id', 'club_value']].copy()
    cv_last.columns = ['user_id', 'cv_last']
    cv_decline = cv_first.merge(cv_last, on='user_id')
    cv_decline['cv_decline_pct'] = (cv_decline['cv_first'] - cv_decline['cv_last']) / cv_decline['cv_first'] * 100
    cv_decline = cv_decline[['user_id', 'cv_decline_pct']]

    print("  [2-4] 거래 활동 피처 생성 중 (관측 기간만)...")

    trade_count = obs_trades.groupby('user_id').size().reset_index(name='obs_trade_count')

    # ── 전체 피처 병합 ──
    print("  [2-5] 피처 병합 중...")
    features = features.merge(last_login_outcome[['user_id', 'is_churned']],
                              on='user_id', how='left')
    features = features.merge(login_days, on='user_id', how='left')
    features = features.merge(avg_session, on='user_id', how='left')
    features = features.merge(early_cnt, on='user_id', how='left')
    features = features.merge(late_cnt, on='user_id', how='left')
    features = features.merge(last_week_cnt, on='user_id', how='left')
    features = features.merge(pkg_count, on='user_id', how='left')
    features = features.merge(pkg_amount, on='user_id', how='left')
    features = features.merge(pkg_variety, on='user_id', how='left')
    features = features.merge(pkg_ovr, on='user_id', how='left')
    features = features.merge(cv_obs_index, on='user_id', how='left')
    features = features.merge(cv_max_drop, on='user_id', how='left')
    features = features.merge(shock_count, on='user_id', how='left')
    features = features.merge(cv_avg_change, on='user_id', how='left')
    features = features.merge(cv_decline, on='user_id', how='left')
    features = features.merge(trade_count, on='user_id', how='left')

    # 결측값 처리
    fill_zero_cols = ['obs_login_days', 'obs_avg_session', 'early_logins',
                      'late_logins', 'last_week_logins', 'obs_pkg_count',
                      'obs_pkg_amount', 'obs_pkg_variety', 'obs_avg_pkg_ovr',
                      'shock_count', 'obs_trade_count', 'cv_decline_pct',
                      'pkg1_cv_change', 'pkg2_cv_change', 'pkg_cumulative_shock',
                      'pkg1_pre_idx', 'pkg1_post_idx', 'pkg2_pre_idx', 'pkg2_post_idx']
    for col in fill_zero_cols:
        features[col] = features[col].fillna(0)

    features['cv_obs_index'] = features['cv_obs_index'].fillna(100)
    features['cv_max_daily_drop'] = features['cv_max_daily_drop'].fillna(0)
    features['cv_avg_change_rate'] = features['cv_avg_change_rate'].fillna(0)
    features['is_churned'] = features['is_churned'].fillna(1)

    # 파생 피처: 접속 감소율 (전반부 대비 후반부)
    features['login_decline_rate'] = np.where(
        features['early_logins'] > 0,
        (features['early_logins'] - features['late_logins']) / features['early_logins'],
        0
    ).clip(-1, 1)

    print(f"\n  → 최종 피처 데이터: {features.shape[0]:,}행 × {features.shape[1]}열")
    print(f"  → 이탈률: {features['is_churned'].mean()*100:.1f}%")
    print(f"  → 그룹별 이탈률:")
    for g in GROUP_ORDER:
        rate = features[features['club_value_group'] == g]['is_churned'].mean() * 100
        print(f"    {g}: {rate:.1f}%")

    return features


# ============================================================================
# 3. 모델 학습 및 평가
# ============================================================================
def train_and_evaluate(features):
    """
    Random Forest + Logistic Regression 학습 및 비교
    """
    print("\n[3] 모델 학습 및 평가")

    # =========================================================================
    # 피처 선택 — EDA 가설 근거 시각화(fig_a1~a3)에서 증명된 상관관계 및 연관성 기반
    # =========================================================================
    # [순서] EDA 시각화로 가설 증명 → 증명된 관계에서 피처 추출 → ML 모델 학습
    #
    # ★ 가설 핵심 피처 (fig_a1/a2/a3에서 통계적으로 유의한 관계 확인됨):
    #   - fig_a1: OVR 130~134 범위 유저일수록 구단가치 하락 큼
    #     → ovr_pkg_overlap, is_pkg_ovr_range
    #   - fig_a2: 패키지 출시 전후 10~100조 그룹 하락폭 가장 큼 (17.3pt)
    #     → pkg1_cv_change, pkg2_cv_change, pkg_cumulative_shock
    #   - fig_a3: OVR가격하락률 → 구단가치하락률 양의 상관 (추세선 기울기=10.1)
    #     → ovr_price_exposure
    #
    # 추가: 일반적 이탈 예측 지표 (접속 패턴, 구매 행동, 구단가치 변동)
    # =========================================================================
    feature_cols = [
        # ── 유저 프로필 ──
        'club_value_jo',       # 구단가치 (조 단위)
        'avg_ovr',             # 평균 OVR
        'spendig',             # 누적 과금
        'membership_rank',     # 멤버십 등급 (숫자)
        # ── 접속 패턴 ──
        'obs_login_days',      # 관측기간 접속일수
        'obs_avg_session',     # 관측기간 평균 세션
        'early_logins',        # 전반부(1월) 접속수
        'late_logins',         # 후반부(2월) 접속수
        'last_week_logins',    # 관측 마지막주 접속수
        'login_decline_rate',  # 접속 감소율
        # ── 구매 행동 ──
        'obs_pkg_count',       # 관측기간 패키지 구매 횟수
        'obs_pkg_amount',      # 관측기간 총 구매 금액
        'obs_pkg_variety',     # 구매 패키지 종류 수
        'obs_avg_pkg_ovr',     # 평균 구매 OVR
        # ★ 가설 핵심 피처 (EDA fig_a1/a2/a3에서 인과관계 증명됨)
        'ovr_pkg_overlap',     # [fig_a1] 유저OVR↔패키지OVR 겹침도
        'is_pkg_ovr_range',    # [fig_a1] 패키지 높은확률 OVR 범위(130~134) 해당여부
        'pkg1_cv_change',      # [fig_a2] 1차 패키지(1/18) 전후 구단가치 변화율
        'pkg2_cv_change',      # [fig_a2] 2차 패키지(2/20) 전후 구단가치 변화율
        'pkg_cumulative_shock',# [fig_a2] 1차+2차 패키지 누적 충격
        'ovr_price_exposure',  # [fig_a3] 보유 OVR 가격 하락 노출도
        # ── 구단가치 변동 ──
        'cv_obs_index',        # 관측기간 말 구단가치 인덱스
        'cv_max_daily_drop',   # 최대 일일 하락률
        'shock_count',         # z-score 충격 횟수
        'cv_avg_change_rate',  # 평균 일일 변동률
        'cv_decline_pct',      # 구단가치 총 하락률(%)
        # ── 거래 ──
        'obs_trade_count',     # 관측기간 거래 횟수
    ]

    # 피처 이름 한글 매핑 (시각화용)
    feature_names_kr = {
        'club_value_jo': '구단가치(조)',
        'avg_ovr': '평균 OVR',
        'spendig': '누적과금',
        'membership_rank': '멤버십등급',
        'obs_login_days': '접속일수(관측기간)',
        'obs_avg_session': '평균세션(분)',
        'early_logins': '1월접속수',
        'late_logins': '2월접속수',
        'last_week_logins': '마지막주접속',
        'login_decline_rate': '접속감소율',
        'obs_pkg_count': '패키지구매횟수',
        'obs_pkg_amount': '총구매금액',
        'obs_pkg_variety': '패키지종류수',
        'obs_avg_pkg_ovr': '구매패키지OVR',
        'ovr_pkg_overlap': 'OVR↔패키지겹침도★',
        'is_pkg_ovr_range': '패키지OVR범위해당★',
        'pkg1_cv_change': '1차패키지후구단가치변화★',
        'pkg2_cv_change': '2차패키지후구단가치변화★',
        'pkg_cumulative_shock': '패키지누적충격★',
        'ovr_price_exposure': 'OVR가격하락노출도★',
        'cv_obs_index': '구단가치인덱스',
        'cv_max_daily_drop': '최대일일하락률',
        'shock_count': 'Z-score충격횟수',
        'cv_avg_change_rate': '평균일일변동률',
        'cv_decline_pct': '구단가치하락률(%)',
        'obs_trade_count': '거래횟수',
    }

    X = features[feature_cols].copy()
    y = features['is_churned'].astype(int)

    # 학습/테스트 분할 (80/20, 층화추출)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y)

    print(f"  → 학습 세트: {len(X_train):,}개 (이탈 {y_train.mean()*100:.1f}%)")
    print(f"  → 테스트 세트: {len(X_test):,}개 (이탈 {y_test.mean()*100:.1f}%)")

    # 스케일링 (Logistic Regression용)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ── Random Forest ──
    print("\n  [3-1] Random Forest 학습 중...")
    rf = RandomForestClassifier(
        n_estimators=200,       # 충분한 트리 수
        max_depth=12,           # 과적합 방지
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight='balanced',  # 클래스 불균형 보정
        random_state=SEED,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_prob = rf.predict_proba(X_test)[:, 1]

    # RF 교차검증
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    rf_cv_scores = cross_val_score(rf, X, y, cv=cv, scoring='f1')

    print(f"    Accuracy:  {accuracy_score(y_test, rf_pred):.4f}")
    print(f"    Precision: {precision_score(y_test, rf_pred):.4f}")
    print(f"    Recall:    {recall_score(y_test, rf_pred):.4f}")
    print(f"    F1:        {f1_score(y_test, rf_pred):.4f}")
    print(f"    AUC:       {roc_auc_score(y_test, rf_prob):.4f}")
    print(f"    CV F1:     {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}")

    # ── Logistic Regression ──
    print("\n  [3-2] Logistic Regression 학습 중...")
    lr = LogisticRegression(
        C=1.0,
        class_weight='balanced',
        max_iter=1000,
        random_state=SEED
    )
    lr.fit(X_train_scaled, y_train)
    lr_pred = lr.predict(X_test_scaled)
    lr_prob = lr.predict_proba(X_test_scaled)[:, 1]

    # LR 교차검증
    lr_cv_scores = cross_val_score(lr, scaler.transform(X), y, cv=cv, scoring='f1')

    print(f"    Accuracy:  {accuracy_score(y_test, lr_pred):.4f}")
    print(f"    Precision: {precision_score(y_test, lr_pred):.4f}")
    print(f"    Recall:    {recall_score(y_test, lr_pred):.4f}")
    print(f"    F1:        {f1_score(y_test, lr_pred):.4f}")
    print(f"    AUC:       {roc_auc_score(y_test, lr_prob):.4f}")
    print(f"    CV F1:     {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")

    results = {
        'rf': rf, 'lr': lr, 'scaler': scaler,
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'rf_pred': rf_pred, 'rf_prob': rf_prob,
        'lr_pred': lr_pred, 'lr_prob': lr_prob,
        'feature_cols': feature_cols,
        'feature_names_kr': feature_names_kr,
        'rf_cv': rf_cv_scores, 'lr_cv': lr_cv_scores,
    }

    return results


# ============================================================================
# 4. 시각화 생성
# ============================================================================
def save_fig(fig, filename):
    """차트 저장 — 마운트 경로 길이 제한 우회 (짧은 이름 저장 후 rename)"""
    final_path = os.path.join(OUTPUT_PATH, filename)
    tmp_name = os.path.join(OUTPUT_PATH, '_t.png')
    fig.savefig(tmp_name, bbox_inches='tight', facecolor='white')
    try:
        os.remove(final_path)
    except OSError:
        pass
    os.rename(tmp_name, final_path)
    plt.show()  # 노트북 인라인 출력
    plt.close(fig)
    print(f"  → 저장: {filename}")


def plot_feature_importance(results):
    """Fig 16: Random Forest Feature Importance"""
    print("\n[Fig 16] Feature Importance")

    rf = results['rf']
    feature_cols = results['feature_cols']
    names_kr = results['feature_names_kr']

    importances = rf.feature_importances_
    indices = np.argsort(importances)[::-1]

    fig, ax = plt.subplots(figsize=(12, 8))

    # 상위 15개 피처 (가로 바차트)
    top_n = min(15, len(feature_cols))
    top_idx = indices[:top_n][::-1]  # 역순 (아래→위)

    y_pos = range(top_n)
    bars = ax.barh(y_pos, importances[top_idx], color='#2196F3', edgecolor='white')

    # 가장 중요한 피처 강조
    bars[-1].set_color('#e74c3c')
    bars[-2].set_color('#f39c12')
    bars[-3].set_color('#f39c12')

    ax.set_yticks(y_pos)
    ax.set_yticklabels([names_kr.get(feature_cols[i], feature_cols[i])
                        for i in top_idx], fontsize=11)

    # 값 표시
    for bar, val in zip(bars, importances[top_idx]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

    ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
    ax.set_title('Random Forest — 이탈 예측 피처 중요도 Top 15', fontsize=14)
    ax.grid(axis='x', alpha=0.3)

    save_fig(fig, 'fig_16_feature_importance.png')


def plot_roc_curves(results):
    """Fig 17: ROC 곡선 비교 (RF vs LR)"""
    print("\n[Fig 17] ROC 곡선")

    y_test = results['y_test']

    fig, ax = plt.subplots(figsize=(9, 8))

    # RF ROC
    rf_fpr, rf_tpr, _ = roc_curve(y_test, results['rf_prob'])
    rf_auc = roc_auc_score(y_test, results['rf_prob'])
    ax.plot(rf_fpr, rf_tpr, color='#e74c3c', linewidth=2.5,
            label=f'Random Forest (AUC = {rf_auc:.4f})')

    # LR ROC
    lr_fpr, lr_tpr, _ = roc_curve(y_test, results['lr_prob'])
    lr_auc = roc_auc_score(y_test, results['lr_prob'])
    ax.plot(lr_fpr, lr_tpr, color='#3498db', linewidth=2.5,
            label=f'Logistic Regression (AUC = {lr_auc:.4f})')

    # 대각선 (무작위 분류기)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)

    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC 곡선 비교 — Random Forest vs Logistic Regression', fontsize=14)
    ax.legend(fontsize=12, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

    save_fig(fig, 'fig_17_roc_curves.png')


def plot_confusion_matrices(results):
    """Fig 18: Confusion Matrix (RF + LR 나란히)"""
    print("\n[Fig 18] Confusion Matrix")

    y_test = results['y_test']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    for ax, pred, title in [(ax1, results['rf_pred'], 'Random Forest'),
                             (ax2, results['lr_pred'], 'Logistic Regression')]:
        cm = confusion_matrix(y_test, pred)
        # 비율로 표시
        cm_pct = cm / cm.sum() * 100

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['유지', '이탈'], yticklabels=['유지', '이탈'],
                    annot_kws={'fontsize': 14})

        # 비율 추가
        for i in range(2):
            for j in range(2):
                ax.text(j + 0.5, i + 0.75, f'({cm_pct[i, j]:.1f}%)',
                        ha='center', va='center', fontsize=9, color='gray')

        ax.set_xlabel('예측', fontsize=12)
        ax.set_ylabel('실제', fontsize=12)
        ax.set_title(f'{title}', fontsize=13)

    fig.suptitle('Confusion Matrix — 이탈 예측 정확도', fontsize=15, fontweight='bold', y=1.02)
    fig.tight_layout()
    save_fig(fig, 'fig_18_confusion_matrix.png')


def plot_model_comparison(results):
    """Fig 19: 모델 성능 비교 (레이더 차트 스타일 → 바차트)"""
    print("\n[Fig 19] 모델 성능 비교")

    y_test = results['y_test']

    metrics = {
        'Accuracy': [accuracy_score(y_test, results['rf_pred']),
                     accuracy_score(y_test, results['lr_pred'])],
        'Precision': [precision_score(y_test, results['rf_pred']),
                      precision_score(y_test, results['lr_pred'])],
        'Recall': [recall_score(y_test, results['rf_pred']),
                   recall_score(y_test, results['lr_pred'])],
        'F1': [f1_score(y_test, results['rf_pred']),
               f1_score(y_test, results['lr_pred'])],
        'AUC': [roc_auc_score(y_test, results['rf_prob']),
                roc_auc_score(y_test, results['lr_prob'])],
        'CV F1': [results['rf_cv'].mean(), results['lr_cv'].mean()],
    }

    fig, ax = plt.subplots(figsize=(12, 7))

    metric_names = list(metrics.keys())
    x = np.arange(len(metric_names))
    width = 0.35

    rf_vals = [metrics[m][0] for m in metric_names]
    lr_vals = [metrics[m][1] for m in metric_names]

    bars1 = ax.bar(x - width/2, rf_vals, width, label='Random Forest',
                   color='#e74c3c', alpha=0.85)
    bars2 = ax.bar(x + width/2, lr_vals, width, label='Logistic Regression',
                   color='#3498db', alpha=0.85)

    for bar, val in zip(bars1, rf_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
    for bar, val in zip(bars2, lr_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('모델 성능 비교 — Random Forest vs Logistic Regression', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names, fontsize=11)
    ax.set_ylim([0, 1.1])
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)

    save_fig(fig, 'fig_19_model_comparison.png')


def plot_group_accuracy(results, features):
    """Fig 20: 그룹별 예측 정확도"""
    print("\n[Fig 20] 그룹별 예측 정확도")

    # 테스트 세트의 그룹 정보
    test_idx = results['X_test'].index
    test_groups = features.loc[test_idx, 'club_value_group']
    y_test = results['y_test']
    rf_pred = results['rf_pred']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # 그룹별 정확도 (RF)
    group_acc = {}
    group_recall = {}
    for g in GROUP_ORDER:
        mask = test_groups == g
        if mask.sum() > 0:
            group_acc[g] = accuracy_score(y_test[mask], rf_pred[mask])
            # 이탈자가 있는 경우만 Recall 계산
            if y_test[mask].sum() > 0:
                group_recall[g] = recall_score(y_test[mask], rf_pred[mask])
            else:
                group_recall[g] = 0

    colors = [COLORS[g] for g in GROUP_ORDER]

    # Accuracy
    bars1 = ax1.bar(GROUP_ORDER, [group_acc.get(g, 0) for g in GROUP_ORDER],
                    color=colors, edgecolor='white')
    for bar, val in zip(bars1, [group_acc.get(g, 0) for g in GROUP_ORDER]):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_title('그룹별 RF 예측 정확도', fontsize=13)
    ax1.set_ylim([0, 1.1])
    ax1.grid(axis='y', alpha=0.3)

    # Recall (이탈 탐지율)
    bars2 = ax2.bar(GROUP_ORDER, [group_recall.get(g, 0) for g in GROUP_ORDER],
                    color=colors, edgecolor='white')
    for bar, val in zip(bars2, [group_recall.get(g, 0) for g in GROUP_ORDER]):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax2.set_ylabel('Recall (이탈 탐지율)', fontsize=12)
    ax2.set_title('그룹별 RF 이탈 탐지율', fontsize=13)
    ax2.set_ylim([0, 1.1])
    ax2.grid(axis='y', alpha=0.3)

    fig.suptitle('Random Forest — 구단가치 그룹별 예측 성능', fontsize=15,
                 fontweight='bold', y=1.02)
    fig.tight_layout()
    save_fig(fig, 'fig_20_group_accuracy.png')


def plot_lr_coefficients(results):
    """Fig 21: Logistic Regression 계수 (영향력 방향 포함)"""
    print("\n[Fig 21] LR 계수")

    lr = results['lr']
    feature_cols = results['feature_cols']
    names_kr = results['feature_names_kr']

    coefs = lr.coef_[0]
    sorted_idx = np.argsort(np.abs(coefs))[::-1]

    fig, ax = plt.subplots(figsize=(12, 8))

    top_n = min(15, len(feature_cols))
    top_idx = sorted_idx[:top_n][::-1]

    y_pos = range(top_n)
    colors_bar = ['#e74c3c' if coefs[i] > 0 else '#3498db' for i in top_idx]

    bars = ax.barh(y_pos, coefs[top_idx], color=colors_bar, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([names_kr.get(feature_cols[i], feature_cols[i])
                        for i in top_idx], fontsize=11)

    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('계수 (양수=이탈 증가, 음수=이탈 감소)', fontsize=12)
    ax.set_title('Logistic Regression — 이탈 예측 계수 (상위 15개)', fontsize=14)
    ax.grid(axis='x', alpha=0.3)

    # 범례
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#e74c3c', label='이탈 증가 요인'),
                       Patch(facecolor='#3498db', label='이탈 감소 요인')]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower right')

    save_fig(fig, 'fig_21_lr_coefficients.png')


def print_classification_reports(results):
    """분류 리포트 출력 및 저장"""
    print("\n[4] Classification Report")

    y_test = results['y_test']

    print("\n── Random Forest ──")
    print(classification_report(y_test, results['rf_pred'],
                                target_names=['유지', '이탈']))

    print("\n── Logistic Regression ──")
    print(classification_report(y_test, results['lr_pred'],
                                target_names=['유지', '이탈']))

    # 리포트 파일 저장
    report_path = os.path.join(MODEL_PATH, 'classification_report.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("=" * 60 + "\n")
        f.write("FC Online 4 — ML 이탈 예측 모델 성능 리포트\n")
        f.write("=" * 60 + "\n\n")
        f.write("■ 피처 선택 근거\n")
        f.write("  EDA 가설 근거 시각화(fig_a1~a3)에서 증명된 인과관계 기반:\n")
        f.write("  - fig_a1: 유저 OVR이 패키지 높은확률 OVR(130~134)과 겹칠수록 구단가치 하락\n")
        f.write("  - fig_a2: 패키지 출시 전후 10~100조 그룹 하락폭 최대 (17.3pt)\n")
        f.write("  - fig_a3: OVR가격하락 → 구단가치하락 → 이탈 인과 체인 확인\n")
        f.write("  → 위 근거로 6개 가설 핵심 피처 + 일반 이탈 지표 선정\n\n")
        f.write("■ 데이터 누출 방지: 시간 분할\n")
        f.write("  관측 기간 (피처): 2025-01-01 ~ 2025-03-01\n")
        f.write("  결과 기간 (타겟): 2025-03-01 ~ 2025-04-30\n\n")

        f.write("── Random Forest ──\n")
        f.write(classification_report(y_test, results['rf_pred'],
                                      target_names=['유지', '이탈']))
        f.write(f"\nAUC: {roc_auc_score(y_test, results['rf_prob']):.4f}\n")
        f.write(f"CV F1: {results['rf_cv'].mean():.4f} ± {results['rf_cv'].std():.4f}\n")

        f.write("\n\n── Logistic Regression ──\n")
        f.write(classification_report(y_test, results['lr_pred'],
                                      target_names=['유지', '이탈']))
        f.write(f"\nAUC: {roc_auc_score(y_test, results['lr_prob']):.4f}\n")
        f.write(f"CV F1: {results['lr_cv'].mean():.4f} ± {results['lr_cv'].std():.4f}\n")

        # 피처 중요도 Top 10
        f.write("\n\n── Random Forest Feature Importance Top 10 ──\n")
        importances = results['rf'].feature_importances_
        indices = np.argsort(importances)[::-1]
        names_kr = results['feature_names_kr']
        feature_cols = results['feature_cols']
        for rank, i in enumerate(indices[:10], 1):
            name = names_kr.get(feature_cols[i], feature_cols[i])
            f.write(f"  {rank}. {name}: {importances[i]:.4f}\n")

    print(f"  → 리포트 저장: {report_path}")


# ============================================================================
# 메인 실행
# ============================================================================
def main():
    # 1. 데이터 로드
    up, ll, pp, tm, dcv = load_data()

    # 2. 피처 엔지니어링
    features = engineer_features(up, ll, pp, tm, dcv)

    # 3. 모델 학습 및 평가
    results = train_and_evaluate(features)

    # 4. 시각화 생성
    plot_feature_importance(results)
    plot_roc_curves(results)
    plot_confusion_matrices(results)
    plot_model_comparison(results)
    plot_group_accuracy(results, features)
    plot_lr_coefficients(results)

    # 5. 리포트 출력
    print_classification_reports(results)

    # 6. 모델 및 파이프라인 저장 (Stage 4에서 로드하여 예측에 사용)
    save_model_pipeline(results, features)

    print("\n" + "=" * 70)
    print("Stage 3 완료! ML 이탈 예측 모델 학습 및 시각화 생성")
    print(f"차트: fig_16 ~ fig_21 ({OUTPUT_PATH})")
    print(f"리포트: {MODEL_PATH}/classification_report.txt")
    print(f"모델: {MODEL_PATH}/rf_model.pkl, scaler.pkl, feature_cols.pkl")
    print("=" * 70)


# ============================================================================
# 5-1. 모델 저장 (Stage 4 시나리오 시뮬레이션에서 사용)
# ============================================================================
def save_model_pipeline(results, features):
    """
    학습된 모델 파이프라인을 저장하여 Stage 4에서 로드 가능하게 함

    저장 항목:
      - rf_model.pkl      : 학습된 Random Forest 모델
      - scaler.pkl         : StandardScaler (LR용)
      - feature_cols.pkl   : 피처 컬럼 목록 (동일 순서 보장)
      - lr_model.pkl       : 학습된 Logistic Regression 모델
      - features_base.pkl  : 전체 유저 피처 데이터 (시나리오별 피처 재계산의 베이스)
    """
    import joblib

    print("\n[5-1] 모델 파이프라인 저장 (Stage 4 시나리오 시뮬레이션용)")

    # RF 모델 저장
    joblib.dump(results['rf'], os.path.join(MODEL_PATH, 'rf_model.pkl'))
    print(f"  → RF 모델 저장: rf_model.pkl")

    # LR 모델 저장
    joblib.dump(results['lr'], os.path.join(MODEL_PATH, 'lr_model.pkl'))
    print(f"  → LR 모델 저장: lr_model.pkl")

    # 스케일러 저장
    joblib.dump(results['scaler'], os.path.join(MODEL_PATH, 'scaler.pkl'))
    print(f"  → 스케일러 저장: scaler.pkl")

    # 피처 컬럼 목록 저장 (순서 유지 중요)
    joblib.dump(results['feature_cols'], os.path.join(MODEL_PATH, 'feature_cols.pkl'))
    print(f"  → 피처 목록 저장: feature_cols.pkl ({len(results['feature_cols'])}개 피처)")

    # 전체 유저 피처 데이터 저장 (시나리오별 피처 재계산의 베이스로 활용)
    # Stage 4에서 OVR을 바꿀 때 ovr_pkg_overlap, is_pkg_ovr_range 등만 재계산하고
    # 나머지 피처(접속패턴, 구매행동 등)는 이 데이터를 그대로 사용
    # 중복 컬럼 제거 (avg_ovr이 feature_cols에도 있을 수 있음)
    extra_cols = ['user_id', 'club_value_group', 'avg_ovr',
                  'monthly_membership_fee', 'monthly_avg_spending']
    base_cols = extra_cols + [c for c in results['feature_cols'] if c not in extra_cols]
    # monthly_membership_fee, monthly_avg_spending이 없으면 user_profile에서 머지
    merge_cols = []
    for col in ['monthly_membership_fee', 'monthly_avg_spending']:
        if col not in features.columns:
            merge_cols.append(col)
    if merge_cols:
        user_profile = pd.read_csv(os.path.join(DATA_PATH, 'user_profile.csv'))
        features = features.merge(
            user_profile[['user_id'] + merge_cols], on='user_id', how='left')
    features_base = features[base_cols].copy()
    joblib.dump(features_base, os.path.join(MODEL_PATH, 'features_base.pkl'))
    print(f"  → 베이스 피처 저장: features_base.pkl ({len(features_base):,}명)")

    print("  ✅ 모델 파이프라인 저장 완료 → Stage 4에서 로드하여 예측에 사용")


main()

In [ ]:
"""
=============================================================================
FC Online 4 데이터 분석 프로젝트 - Stage 4: ML 기반 패키지 OVR 시나리오 시뮬레이션
=============================================================================
목적:
  Stage 3에서 학습·저장한 ML 이탈 예측 모델(Random Forest)을 로드하여,
  패키지의 평균 OVR을 조절했을 때 유저별 이탈 확률이 어떻게 변하는지
  직접 predict_proba()로 예측 → 최적 패키지 OVR 제안

핵심 로직:
  1. Stage 3 저장 파이프라인 로드 (RF 모델 + 스케일러 + 피처목록 + 유저 피처)
  2. 각 OVR 시나리오마다 "제어 가능한 피처"만 재계산:
     - ovr_pkg_overlap: 유저OVR ↔ 새 패키지OVR 중심 거리 기반 겹침도
     - is_pkg_ovr_range: 새 패키지 높은확률 OVR 범위 해당 여부
  3. 나머지 피처(접속패턴, 구매행동, 구단가치변동 등)는 관측 데이터 그대로 유지
  4. model.predict_proba() → 유저별 이탈 확률 → 그룹별/전체 이탈률 집계
  5. 매출/손실 계산 → 최적 OVR 제안

배경 (EDA에서 증명됨):
  - fig_a1: 유저 OVR이 패키지 높은확률 OVR(130~134)과 겹칠수록 구단가치 하락 큼
  - fig_a2: 패키지 출시 전후 10~100조 그룹 하락폭 가장 큼 (17.3pt)
  - fig_a3: OVR가격하락 → 구단가치하락 → 이탈 인과체인 (r=0.872)
  - fig_12/12b: 이탈 손실(135.5억) vs 총 기대수익(502.8억, 패키지+잔존유저)

출력:
  - fig_22: 시나리오별 그룹 이탈률 비교 (Grouped Bar)
  - fig_23: 시나리오별 총 이탈률 + 매출/손실 비교
  - fig_24: 최적 OVR 중심점 탐색 곡선
  - fig_25: 최종 액션 아이템 요약 대시보드
=============================================================================
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import joblib

# 폰트 설정
sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings('ignore')
setup_korean_font()

# ============================================================================
# 경로 설정
# ============================================================================
BASE_DIR = PROJECT_ROOT
DATA_PATH = os.path.join(BASE_DIR, 'data')
OUTPUT_PATH = os.path.join(BASE_DIR, 'outputs', 'figures')
MODEL_PATH = os.path.join(BASE_DIR, 'outputs', 'models')
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 이탈 손실 계산 기간
LOSS_MONTHS = 8  # 이탈 유저의 예상 미래 손실 기간 (개월)
# 월 기대수익은 user_profile의 monthly_membership_fee + monthly_avg_spending에서 직접 참조
# → load_model_pipeline()에서 features_base에 포함

# 패키지 매출 가우시안 파라미터
# 패키지 매력도는 OVR 133 부근에서 최대 (대다수 유저가 원하는 OVR대)
# 너무 낮으면 → 안 사고, 너무 높으면 → 비싸서 못 삼
REVENUE_CENTER = 133  # 매출 최대 OVR
REVENUE_SIGMA = 4.0   # 매출 감소 속도
BASE_PKG_REVENUE = 27.4   # 패키지 판매 매출 최대치 (억)
RETENTION_MONTHS = 8  # 잔존 유저 기대수익 산정 기간 (손실과 동일)

print("=" * 70)
print("FC Online 4 - Stage 4: ML 기반 패키지 OVR 시나리오 시뮬레이션")
print("=" * 70)


# ============================================================================
# 1. Stage 3 모델 파이프라인 로드
# ============================================================================
def load_model_pipeline():
    """
    Stage 3에서 저장한 모델 파이프라인을 로드
    - RF 모델, 스케일러, 피처 목록, 유저 베이스 피처
    """
    print("\n[1] ML 모델 파이프라인 로드 (Stage 3 산출물)")

    rf_model = joblib.load(os.path.join(MODEL_PATH, 'rf_model.pkl'))
    scaler = joblib.load(os.path.join(MODEL_PATH, 'scaler.pkl'))
    feature_cols = joblib.load(os.path.join(MODEL_PATH, 'feature_cols.pkl'))
    features_base = joblib.load(os.path.join(MODEL_PATH, 'features_base.pkl'))

    print(f"  → RF 모델 로드 완료 (n_estimators={rf_model.n_estimators})")
    print(f"  → 피처 목록: {len(feature_cols)}개")
    print(f"  → 유저 베이스 피처: {len(features_base):,}명")

    return rf_model, scaler, feature_cols, features_base


# ============================================================================
# 2. 시나리오 정의
# ============================================================================
def define_scenarios():
    """
    패키지 OVR 중심점을 변경한 시나리오 정의
    각 시나리오는 패키지에서 높은 확률로 등장하는 OVR 범위를 변경
    """
    scenarios = {
        '현행 (130~134)': {
            'center': 132, 'range': (130, 134),
            'description': '현재 패키지 설정 (OVR 130~134 중심)',
            'color': '#e74c3c',
        },
        '하향A (126~130)': {
            'center': 128, 'range': (126, 130),
            'description': '저가 선수 중심으로 대폭 하향',
            'color': '#3498db',
        },
        '소폭하향B (128~132)': {
            'center': 130, 'range': (128, 132),
            'description': '소폭 하향 (현행 대비 OVR -2)',
            'color': '#2ecc71',
        },
        '상향C (134~138)': {
            'center': 136, 'range': (134, 138),
            'description': '고가 선수 중심으로 상향',
            'color': '#f39c12',
        },
        '대폭상향D (136~140)': {
            'center': 138, 'range': (136, 140),
            'description': '최고가 선수 중심으로 대폭 상향',
            'color': '#9b59b6',
        },
    }
    return scenarios


# ============================================================================
# 3. ML 기반 시뮬레이션 엔진
# ============================================================================
def recalculate_scenario_features(features_base, feature_cols, pkg_center, pkg_range):
    """
    새로운 패키지 OVR 시나리오에 맞춰 "제어 가능한 피처"만 재계산

    변경되는 피처 (패키지 OVR에 의존):
      - ovr_pkg_overlap: 유저OVR ↔ 새 패키지OVR 거리 기반 겹침도
      - is_pkg_ovr_range: 새 패키지 높은확률 OVR 범위 해당 여부

    유지되는 피처 (관측 데이터 기반 — 시나리오와 무관):
      - 접속 패턴, 구매 행동, 구단가치 변동, 거래 활동 등
      - pkg1/2_cv_change, pkg_cumulative_shock (이미 발생한 과거 데이터)
      - ovr_price_exposure (관측 기간 내 실제 가격 변동)

    Args:
        features_base: Stage 3에서 저장한 전체 유저 피처 DataFrame
        feature_cols: 모델이 사용하는 피처 컬럼 순서
        pkg_center: 새 패키지 OVR 중심점
        pkg_range: (min_ovr, max_ovr) 새 패키지 높은확률 OVR 범위
    Returns:
        X_scenario: 시나리오용 피처 DataFrame (feature_cols 순서)
    """
    X = features_base[feature_cols].copy()

    # ── (A) ovr_pkg_overlap 재계산 ──
    # 유저 avg_ovr과 새 패키지 OVR 중심 간 거리 → 겹침도
    # Stage 3 원본: 1.0 / (1.0 + abs(avg_ovr - PKG_CENTER_OVR))
    ovr_distance = abs(features_base['avg_ovr'] - pkg_center)
    X['ovr_pkg_overlap'] = 1.0 / (1.0 + ovr_distance)

    # ── (B) is_pkg_ovr_range 재계산 ──
    # 유저 avg_ovr이 새 패키지 높은확률 OVR 범위에 해당하는지
    # Stage 3 원본: avg_ovr.between(129.5, 134.5)
    X['is_pkg_ovr_range'] = features_base['avg_ovr'].between(
        pkg_range[0] - 0.5, pkg_range[1] + 0.5
    ).astype(int)

    return X


def simulate_scenario_ml(rf_model, features_base, feature_cols, scenario_config):
    """
    ML 모델(RF)로 시나리오별 유저 이탈 확률을 직접 예측

    프로세스:
      1. 시나리오에 맞게 피처 재계산
      2. rf_model.predict_proba() → 유저별 이탈 확률
      3. 그룹별 평균 이탈률 + 이탈 유저 수 집계
    """
    pkg_center = scenario_config['center']
    pkg_range = scenario_config['range']

    # 시나리오 피처 재계산
    X_scenario = recalculate_scenario_features(
        features_base, feature_cols, pkg_center, pkg_range)

    # ML 모델로 유저별 이탈 확률 예측
    churn_proba = rf_model.predict_proba(X_scenario)[:, 1]

    # 그룹별 결과 집계
    results = {}
    for group in GROUP_ORDER:
        mask = features_base['club_value_group'] == group
        group_proba = churn_proba[mask]
        n_users = mask.sum()

        if n_users == 0:
            results[group] = {'churn_rate': 0, 'n_churned': 0, 'n_users': 0}
            continue

        # 그룹 평균 이탈 확률 = 예상 이탈률
        avg_churn_rate = group_proba.mean() * 100
        n_churned = int(n_users * group_proba.mean())

        results[group] = {
            'churn_rate': avg_churn_rate,
            'n_churned': n_churned,
            'n_users': n_users,
            'avg_proba': group_proba.mean(),
            'median_proba': np.median(group_proba),
        }

    # 전체 이탈률
    total_churned = sum(r['n_churned'] for r in results.values())
    total_users = sum(r['n_users'] for r in results.values())
    results['전체'] = {
        'churn_rate': churn_proba.mean() * 100,
        'n_churned': total_churned,
        'n_users': total_users,
    }

    return results


def calculate_revenue_loss(scenario_results, pkg_center, features_base):
    """
    시나리오별 총 기대수익 vs 이탈 손실 계산

    총 기대수익 = 패키지 판매 매출 + 잔존(비이탈) 유저의 향후 기대수익
      - 패키지 매출: OVR 133 중심 Gaussian bell curve
      - 잔존 수익: 이탈하지 않은 유저의 (멤버십+과금) × 8개월
    이탈 손실 = 이탈 유저 × (월 멤버십 + 월 과금) × 8개월

    → 순영향 = 총 기대수익 - 이탈 손실
    → OVR 최적화는 "이탈을 줄여 잔존 수익을 높이면서 패키지 매출도 유지"하는 균형점
    """
    # (1) 패키지 판매 매출: 종 모양 곡선 (OVR 133에서 최대)
    rev_factor = np.exp(-0.5 * ((pkg_center - REVENUE_CENTER) / REVENUE_SIGMA) ** 2)
    pkg_revenue = BASE_PKG_REVENUE * rev_factor

    # (2) 잔존 유저 기대수익: 이탈하지 않은 유저의 향후 멤버십+과금 수익
    retention_revenue = 0
    total_loss = 0
    for group in GROUP_ORDER:
        group_mask = features_base['club_value_group'] == group
        n_total = group_mask.sum()
        n_churned = scenario_results[group]['n_churned']
        n_retained = n_total - n_churned

        # 그룹별 평균 월 기대수익 (데이터 기반)
        avg_fee = features_base.loc[group_mask, 'monthly_membership_fee'].mean()
        avg_spending = features_base.loc[group_mask, 'monthly_avg_spending'].mean() \
            if 'monthly_avg_spending' in features_base.columns else 0
        monthly_rev = avg_fee + avg_spending

        # 잔존 유저 수익 (억 단위)
        retention_revenue += n_retained * monthly_rev * RETENTION_MONTHS / 1e8
        # 이탈 유저 손실 (억 단위)
        total_loss += n_churned * monthly_rev * LOSS_MONTHS / 1e8

    # 총 기대수익 = 패키지 매출 + 잔존 유저 수익
    total_revenue = pkg_revenue + retention_revenue

    return total_revenue, total_loss


# ============================================================================
# 4. 전체 시나리오 시뮬레이션 실행
# ============================================================================
def run_all_scenarios(rf_model, features_base, feature_cols):
    """모든 시나리오에 대해 ML 모델 기반 이탈률 예측 및 결과 집계"""
    print("\n[2] ML 모델 기반 시나리오 시뮬레이션 실행")

    scenarios = define_scenarios()
    all_results = {}

    for name, config in scenarios.items():
        print(f"\n  ▶ {name}: {config['description']}")

        # ML 모델로 이탈 확률 예측
        sim_results = simulate_scenario_ml(
            rf_model, features_base, feature_cols, config)

        # 매출/손실 계산
        revenue, loss = calculate_revenue_loss(sim_results, config['center'], features_base)

        all_results[name] = {
            'config': config,
            'sim_results': sim_results,
            'revenue': revenue,
            'loss': loss,
            'net_impact': revenue - loss,
        }

        print(f"    전체 이탈률: {sim_results['전체']['churn_rate']:.1f}% "
              f"(ML predict_proba 기반)")
        print(f"    매출: {revenue:.1f}억 / 손실: {loss:.1f}억 / 순영향: {revenue-loss:+.1f}억")

        for g in GROUP_ORDER:
            r = sim_results[g]
            print(f"    {g}: 이탈률 {r['churn_rate']:.1f}%, "
                  f"이탈 {r['n_churned']:,}명")

    return all_results


# ============================================================================
# 5. 최적 OVR 탐색 (연속 ML 예측)
# ============================================================================
def find_optimal_ovr(rf_model, features_base, feature_cols):
    """
    패키지 OVR 중심점을 125~142까지 1단위로 변경하며
    ML 모델로 이탈률 예측 → 매출/손실/순영향 계산 → 최적점 탐색
    """
    print("\n[3] ML 기반 최적 OVR 중심점 탐색")

    ovr_range = np.arange(125, 143, 1)
    results_list = []

    for center in ovr_range:
        pkg_range = (center - 2, center + 2)
        config = {'center': center, 'range': pkg_range}

        # ML 모델로 예측
        sim = simulate_scenario_ml(rf_model, features_base, feature_cols, config)

        # 매출/손실 계산
        revenue, loss = calculate_revenue_loss(sim, center, features_base)

        results_list.append({
            'ovr_center': center,
            'total_churn_rate': sim['전체']['churn_rate'],
            'revenue': revenue,
            'loss': loss,
            'net_impact': revenue - loss,
            'churn_0_10': sim['0~10조']['churn_rate'],
            'churn_10_100': sim['10~100조']['churn_rate'],
            'churn_100_1000': sim['100~1000조']['churn_rate'],
            'churn_1000_plus': sim['1000조이상']['churn_rate'],
        })

    df = pd.DataFrame(results_list)

    # 최적점: 순영향(매출-손실)이 최대인 OVR
    optimal_idx = df['net_impact'].idxmax()
    optimal_ovr = df.loc[optimal_idx, 'ovr_center']

    print(f"\n  ★ 최적 패키지 OVR 중심점: {optimal_ovr}")
    print(f"    이탈률: {df.loc[optimal_idx, 'total_churn_rate']:.1f}% (ML 예측)")
    print(f"    매출: {df.loc[optimal_idx, 'revenue']:.1f}억")
    print(f"    손실: {df.loc[optimal_idx, 'loss']:.1f}억")
    print(f"    순영향: {df.loc[optimal_idx, 'net_impact']:+.1f}억")

    return df, optimal_ovr


# ============================================================================
# 6. 시각화
# ============================================================================
def save_fig(fig, filename):
    """차트 저장 — 마운트 경로 길이 제한 우회 (짧은 이름 저장 후 rename)"""
    final_path = os.path.join(OUTPUT_PATH, filename)
    tmp_name = os.path.join(OUTPUT_PATH, '_t.png')
    fig.savefig(tmp_name, bbox_inches='tight', facecolor='white', dpi=150)
    try:
        os.remove(final_path)
    except OSError:
        pass
    os.rename(tmp_name, final_path)
    plt.show()  # 노트북 인라인 출력
    plt.close(fig)
    print(f"  → 저장: {filename}")


def fig_22_scenario_churn_comparison(all_results):
    """Fig 22: 시나리오별 그룹 이탈률 비교 (Grouped Bar) — ML 예측 기반"""
    print("\n[Fig 22] 시나리오별 그룹 이탈률 비교 (ML 예측)")

    scenario_names = list(all_results.keys())
    n_scenarios = len(scenario_names)
    n_groups = len(GROUP_ORDER)

    fig, ax = plt.subplots(figsize=(16, 8))

    x = np.arange(n_groups)
    width = 0.15

    for i, name in enumerate(scenario_names):
        churn_rates = [all_results[name]['sim_results'][g]['churn_rate']
                       for g in GROUP_ORDER]
        offset = (i - n_scenarios / 2 + 0.5) * width
        bars = ax.bar(x + offset, churn_rates, width,
                      label=name,
                      color=all_results[name]['config']['color'],
                      alpha=0.85, edgecolor='white')

        # 현행 시나리오만 값 표시
        if '현행' in name:
            for bar, val in zip(bars, churn_rates):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                        f'{val:.1f}%', ha='center', fontsize=8, fontweight='bold',
                        color='red')

    ax.set_xlabel('구단가치 그룹', fontsize=12)
    ax.set_ylabel('ML 예측 이탈률 (%)', fontsize=12)
    ax.set_title('패키지 OVR 시나리오별 그룹 이탈률 비교 (RF model.predict_proba 기반)\n'
                 '→ OVR 중심을 하향 조절하면 10~100조 그룹 이탈률 감소',
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(GROUP_ORDER, fontsize=11)
    ax.legend(fontsize=9, loc='upper right', ncol=2)
    ax.grid(axis='y', alpha=0.3)

    # 핵심 메시지
    ax.text(0.02, 0.95,
            '▼ 패키지 OVR을 하향하면 10~100조 그룹 이탈 감소\n'
            '▲ 패키지 OVR을 상향하면 100~1000조/1000조이상 이탈 증가\n'
            '※ Random Forest predict_proba() 기반 예측',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9),
            va='top')

    save_fig(fig, 'fig_22_scenario_churn.png')


def fig_23_revenue_loss_comparison(all_results):
    """Fig 23: 시나리오별 매출/손실/순영향 비교"""
    print("\n[Fig 23] 시나리오별 매출 vs 손실 비교")

    scenario_names = list(all_results.keys())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    # (좌) 매출 vs 손실 비교
    x = np.arange(len(scenario_names))
    width = 0.35

    revenues = [all_results[n]['revenue'] for n in scenario_names]
    losses = [all_results[n]['loss'] for n in scenario_names]

    bars1 = ax1.bar(x - width/2, revenues, width, label='패키지 매출',
                    color='#2ecc71', alpha=0.85)
    bars2 = ax1.bar(x + width/2, losses, width, label='이탈 손실',
                    color='#e74c3c', alpha=0.85)

    for bar, val in zip(bars1, revenues):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}', ha='center', fontsize=9)
    for bar, val in zip(bars2, losses):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}', ha='center', fontsize=9)

    ax1.set_ylabel('금액 (억 원)', fontsize=12)
    ax1.set_title('시나리오별 매출 vs 이탈 손실', fontsize=13)
    ax1.set_xticks(x)
    ax1.set_xticklabels([n.split(' ')[0] for n in scenario_names],
                        fontsize=9, rotation=15)
    ax1.legend(fontsize=10)
    ax1.grid(axis='y', alpha=0.3)

    # (우) 순영향 (매출 - 손실)
    net_impacts = [all_results[n]['net_impact'] for n in scenario_names]
    bar_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in net_impacts]

    bars3 = ax2.bar(x, net_impacts, 0.6, color=bar_colors, alpha=0.85,
                    edgecolor='white')

    for bar, val in zip(bars3, net_impacts):
        y_pos = bar.get_height() + 0.2 if val >= 0 else bar.get_height() - 0.5
        ax2.text(bar.get_x() + bar.get_width()/2, y_pos,
                 f'{val:+.1f}억', ha='center', fontsize=10, fontweight='bold')

    ax2.axhline(0, color='black', linewidth=0.8)
    ax2.set_ylabel('순영향 (매출 - 손실, 억 원)', fontsize=12)
    ax2.set_title('시나리오별 순영향 (ML 예측 기반)\n(양수 = 이득, 음수 = 손해)',
                  fontsize=13)
    ax2.set_xticks(x)
    ax2.set_xticklabels([n.split(' ')[0] for n in scenario_names],
                        fontsize=9, rotation=15)
    ax2.grid(axis='y', alpha=0.3)

    fig.suptitle('패키지 OVR 조절에 따른 수익성 분석 (ML 이탈 예측 기반)',
                 fontsize=15, fontweight='bold', y=1.02)
    fig.tight_layout()
    save_fig(fig, 'fig_23_revenue_loss_scenario.png')


def fig_24_optimal_ovr_curve(optimal_df, optimal_ovr):
    """Fig 24: 최적 OVR 탐색 곡선 — ML 예측 기반"""
    print("\n[Fig 24] 최적 OVR 탐색 곡선 (ML 예측)")

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

    df = optimal_df

    # (상) 이탈률 곡선 — 그룹별 (ML 예측)
    ax1.plot(df['ovr_center'], df['churn_10_100'], '-o', color=COLORS['10~100조'],
             linewidth=2.5, markersize=6, label='10~100조 (핵심 그룹)')
    ax1.plot(df['ovr_center'], df['churn_0_10'], '-s', color=COLORS['0~10조'],
             linewidth=1.5, markersize=4, label='0~10조')
    ax1.plot(df['ovr_center'], df['churn_100_1000'], '-^', color=COLORS['100~1000조'],
             linewidth=1.5, markersize=4, label='100~1000조')
    ax1.plot(df['ovr_center'], df['churn_1000_plus'], '-d', color=COLORS['1000조이상'],
             linewidth=1.5, markersize=4, label='1000조이상')

    # 현행 위치 표시
    ax1.axvline(132, color='red', linestyle='--', alpha=0.5, linewidth=1.5)
    ax1.text(132.2, ax1.get_ylim()[1] if ax1.get_ylim()[1] > 0 else 25,
             '현행\n(132)', color='red', fontsize=10, fontweight='bold')

    # 최적 위치 표시
    ax1.axvline(optimal_ovr, color='green', linestyle='--', alpha=0.5, linewidth=1.5)
    ax1.text(optimal_ovr + 0.2, ax1.get_ylim()[1] if ax1.get_ylim()[1] > 0 else 20,
             f'최적\n({int(optimal_ovr)})', color='green', fontsize=10, fontweight='bold')

    ax1.set_ylabel('ML 예측 이탈률 (%)', fontsize=12)
    ax1.set_title('패키지 OVR 중심점에 따른 그룹별 이탈률 변화 (RF predict_proba 기반)',
                  fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10, loc='upper left')
    ax1.grid(True, alpha=0.3)

    # (하) 순영향 곡선
    ax2.fill_between(df['ovr_center'], df['net_impact'], 0,
                     where=(df['net_impact'] >= 0),
                     color='#2ecc71', alpha=0.3, label='순이익 구간')
    ax2.fill_between(df['ovr_center'], df['net_impact'], 0,
                     where=(df['net_impact'] < 0),
                     color='#e74c3c', alpha=0.3, label='순손실 구간')
    ax2.plot(df['ovr_center'], df['net_impact'], '-o', color='#2c3e50',
             linewidth=2.5, markersize=6)

    # 매출선과 손실선
    ax2.plot(df['ovr_center'], df['revenue'], '--', color='#2ecc71',
             linewidth=1.5, alpha=0.7, label='매출')
    ax2.plot(df['ovr_center'], df['loss'], '--', color='#e74c3c',
             linewidth=1.5, alpha=0.7, label='손실')

    ax2.axhline(0, color='black', linewidth=0.8)
    ax2.axvline(132, color='red', linestyle='--', alpha=0.5, linewidth=1.5)
    ax2.axvline(optimal_ovr, color='green', linestyle='--', alpha=0.5, linewidth=1.5)

    # 최적점 강조
    opt_row = df[df['ovr_center'] == optimal_ovr].iloc[0]
    ax2.scatter(optimal_ovr, opt_row['net_impact'], s=200, color='green',
                zorder=5, edgecolors='black', linewidths=2)
    ax2.annotate(f'최적: OVR {int(optimal_ovr)}\n순영향: {opt_row["net_impact"]:+.1f}억',
                 xy=(optimal_ovr, opt_row['net_impact']),
                 xytext=(20, 20), textcoords='offset points',
                 fontsize=11, fontweight='bold', color='green',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9),
                 arrowprops=dict(arrowstyle='->', color='green', lw=2))

    ax2.set_xlabel('패키지 OVR 중심점', fontsize=12)
    ax2.set_ylabel('금액 (억 원)', fontsize=12)
    ax2.set_title('패키지 OVR 중심점에 따른 매출·손실·순영향 (ML 기반)',
                  fontsize=14, fontweight='bold')
    ax2.legend(fontsize=9, loc='upper left', ncol=2)
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    save_fig(fig, 'fig_24_optimal_ovr_curve.png')


def fig_25_action_item_dashboard(all_results, optimal_df, optimal_ovr):
    """Fig 25: 최종 액션 아이템 요약 대시보드"""
    print("\n[Fig 25] 최종 액션 아이템 대시보드")

    fig = plt.figure(figsize=(18, 12))
    gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

    # ── (1) 현행 vs 최적 이탈률 비교 ──
    ax1 = fig.add_subplot(gs[0, 0])
    current = all_results['현행 (130~134)']

    current_rates = [current['sim_results'][g]['churn_rate'] for g in GROUP_ORDER]

    # 최적 시나리오 결과 (optimal_df에서)
    opt_row = optimal_df[optimal_df['ovr_center'] == optimal_ovr].iloc[0]
    optimal_rates = [opt_row['churn_0_10'], opt_row['churn_10_100'],
                     opt_row['churn_100_1000'], opt_row['churn_1000_plus']]

    x = np.arange(len(GROUP_ORDER))
    width = 0.35
    ax1.bar(x - width/2, current_rates, width, label='현행 (132)', color='#e74c3c', alpha=0.8)
    ax1.bar(x + width/2, optimal_rates, width, label=f'제안 ({int(optimal_ovr)})',
            color='#2ecc71', alpha=0.8)

    for i, (c, o) in enumerate(zip(current_rates, optimal_rates)):
        diff = o - c
        color = '#2ecc71' if diff < 0 else '#e74c3c'
        ax1.text(i, max(c, o) + 0.5, f'{diff:+.1f}%p', ha='center',
                 fontsize=9, fontweight='bold', color=color)

    ax1.set_ylabel('ML 예측 이탈률 (%)')
    ax1.set_title('그룹별 이탈률: 현행 vs 제안\n(RF predict_proba 기반)',
                  fontsize=12, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(['0~10조', '10~100조', '100~1000조', '1000조+'], fontsize=8)
    ax1.legend(fontsize=9)
    ax1.grid(axis='y', alpha=0.3)

    # ── (2) 매출/손실 비교 ──
    ax2 = fig.add_subplot(gs[0, 1])

    current_rev = current['revenue']
    current_loss = current['loss']
    opt_rev = opt_row['revenue']
    opt_loss = opt_row['loss']

    categories = ['현행\n매출', '현행\n손실', '제안\n매출', '제안\n손실']
    values = [current_rev, current_loss, opt_rev, opt_loss]
    colors_bar = ['#27ae60', '#c0392b', '#2ecc71', '#e74c3c']

    bars = ax2.bar(categories, values, color=colors_bar, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}억', ha='center', fontsize=10, fontweight='bold')

    ax2.set_ylabel('금액 (억 원)')
    ax2.set_title('매출/손실 비교', fontsize=12, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    # ── (3) 핵심 KPI 변화 ──
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.axis('off')

    current_total_churn = current['sim_results']['전체']['churn_rate']
    optimal_total_churn = opt_row['total_churn_rate']
    churn_diff = optimal_total_churn - current_total_churn

    net_current = current_rev - current_loss
    net_optimal = opt_rev - opt_loss
    net_diff = net_optimal - net_current

    kpi_text = (
        f"━━━ 핵심 KPI 변화 ━━━\n\n"
        f"패키지 OVR 중심점\n"
        f"  현행: 132  →  제안: {int(optimal_ovr)}\n\n"
        f"전체 이탈률 (ML 예측)\n"
        f"  {current_total_churn:.1f}%  →  {optimal_total_churn:.1f}%  ({churn_diff:+.1f}%p)\n\n"
        f"10~100조 이탈률 (핵심)\n"
        f"  {current['sim_results']['10~100조']['churn_rate']:.1f}%  →  "
        f"{opt_row['churn_10_100']:.1f}%\n\n"
        f"순영향 (매출-손실)\n"
        f"  {net_current:+.1f}억  →  {net_optimal:+.1f}억  ({net_diff:+.1f}억)"
    )

    ax3.text(0.1, 0.95, kpi_text, transform=ax3.transAxes,
             fontsize=11, va='top',
             bbox=dict(boxstyle='round', facecolor='#f0f8ff', edgecolor='#3498db',
                       alpha=0.9))
    ax3.set_title('핵심 지표 변화', fontsize=12, fontweight='bold')

    # ── (4) 인과 체인 요약 (텍스트) ──
    ax4 = fig.add_subplot(gs[1, :])
    ax4.axis('off')

    summary_text = (
        "■ 분석 결론 및 액션 아이템\n\n"
        "  [인과 체인] 패키지 OVR 확률분포 → 해당 OVR대 선수 공급과잉 → 선수 가격 하락 → 구단가치 하락 → 유저 이탈\n\n"
        "  [핵심 발견]\n"
        "  • EDA(fig_a1~a3): 유저 보유 OVR이 패키지 높은확률 OVR(130~134)과 겹칠수록 구단가치 하락이 크고, 10~100조 그룹이 가장 민감 (r=0.872)\n"
        "  • ML 모델: OVR 겹침도·패키지 충격·구단가치 하락률이 이탈 예측의 유의한 피처로 확인 (RF AUC=0.73)\n"
        "  • 시나리오 시뮬레이션: 학습된 RF 모델의 predict_proba()로 유저별 이탈 확률 직접 예측 → 시나리오별 비교\n"
        "  • 현행 패키지: 총 기대수익(패키지+잔존유저) 502.8억 vs 이탈 손실 135.5억 → 순영향 +367.2억\n\n"
        f"  [제안] 패키지 높은확률 OVR 중심을 132 → {int(optimal_ovr)}로 조절\n"
        f"  • 10~100조 그룹(전체 54%)의 OVR 겹침도 감소 → 이탈 확률 저하 (ML 예측)\n"
        f"  • 순영향: {net_current:+.1f}억 → {net_optimal:+.1f}억 (개선 {net_diff:+.1f}억)\n"
        f"  • 패키지 매출 소폭 감소를 감수하되, 이탈 방지를 통한 장기 수익 확보\n\n"
        "  [향후 과제]\n"
        "  • 실제 운영 데이터로 ML 예측 결과 검증 (A/B 테스트)\n"
        "  • 패키지 OVR 분포를 균등화하는 방안 검토 (특정 OVR 쏠림 방지)\n"
        "  • 이탈 고위험 유저 대상 맞춤 리텐션 프로그램 설계"
    )

    ax4.text(0.02, 0.95, summary_text, transform=ax4.transAxes,
             fontsize=11, va='top',
             bbox=dict(boxstyle='round', facecolor='#fffde7', edgecolor='#f57f17',
                       alpha=0.9, linewidth=2))

    fig.suptitle('FC Online 4 — ML 기반 패키지 OVR 최적화 제안 대시보드',
                 fontsize=16, fontweight='bold', y=1.01)

    save_fig(fig, 'fig_25_action_dashboard.png')


# ============================================================================
# 메인 실행
# ============================================================================
def main():
    # 1. ML 모델 파이프라인 로드
    rf_model, scaler, feature_cols, features_base = load_model_pipeline()

    # 2. 시나리오 시뮬레이션 (ML 예측)
    all_results = run_all_scenarios(rf_model, features_base, feature_cols)

    # 3. 최적 OVR 탐색 (ML 예측)
    optimal_df, optimal_ovr = find_optimal_ovr(rf_model, features_base, feature_cols)

    # 4. 시각화 생성
    fig_22_scenario_churn_comparison(all_results)
    fig_23_revenue_loss_comparison(all_results)
    fig_24_optimal_ovr_curve(optimal_df, optimal_ovr)
    fig_25_action_item_dashboard(all_results, optimal_df, optimal_ovr)

    print("\n" + "=" * 70)
    print("Stage 4 완료! ML 기반 패키지 OVR 시나리오 시뮬레이션")
    print(f"  예측 방식: Random Forest predict_proba()")
    print(f"  최적 패키지 OVR 중심점: {int(optimal_ovr)}")
    print(f"  차트: fig_22 ~ fig_25 ({OUTPUT_PATH})")
    print("=" * 70)


main()